# Mapping Unique Author Initials to Full Names — All Rules and ZIP Execution

This notebook applies all alias-generation and collision-resolution rules, but the final output is intentionally simple and readable:

- All initials found in the `authors` list are flattened into one **unique set per article**.
- Each unique initial is resolved only once.
- The main CSV contains **one row per unique full name**, not one row per author group.
- Every full name receives its original zero-based index or indices from the `full name` list.
- If the same full name appears more than once, all of its indices are stored in the same row.
- If one full name is observed through multiple initials, all initials are stored in the same row.


## 1. Import Libraries and Define Constants

- `PREFIX_TOKENS`: surname particles such as `van`, `de`, and `von`.
- `COMMON_WORD_ALIAS_REQUIRES_UPPERCASE`: ordinary words that may be treated as initials only when every letter is uppercase.
- `DECL_RE`: removes general declaration text beginning with expressions such as `All authors` or `The authors`.
- `TOKEN_RE`: detects initials, words, and hyphenated names.


In [ ]:
import ast
import csv
import gc
import itertools
import json
import re
import shutil
import time
import unicodedata
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display

PREFIX_TOKENS = {
    "van", "von", "de", "del", "da", "di",
    "la", "le", "du", "der", "den"
}

COMMON_WORD_ALIAS_REQUIRES_UPPERCASE = {
    "the", "all", "has", "had", "and", "for", "from", "with",
    "was", "were", "are", "but", "not", "his", "her", "him",
    "our", "out", "into", "onto", "over", "under", "then", "than",
    "that", "this", "these", "those", "their", "there"
}

STOP_WORDS = {
    "and", "or", "both", "authors", "author", "have", "has", "had",
    "read", "confirm", "confirms", "that", "they", "he", "she",
    "meet", "meets", "icmje", "criteria", "for", "authorship", "the",
    "made", "following", "declarations", "about", "their",
    "contributions", "contribution", "all", "with", "from", "was",
    "were", "are", "but", "not", "his", "her", "him", "our", "out",
    "into", "onto", "over", "under", "then", "than", "this", "these",
    "those", "there"
}

DECL_RE = re.compile(
    r"(?i)\b(?:all\s+authors?|both\s+authors?|the\s+authors?|"
    r"the\s+author|author\(s\))\b.*$"
)

TOKEN_RE = re.compile(
    r"(?:[A-Za-z]\.){1,8}|"
    r"[A-Za-z]+(?:[-–—][A-Za-z]+)+\.?|"
    r"[A-Za-z]+\.?'"
    .replace("\\.?'", "\\.?")
)


## Rule 1 — Normalize Spaces, Symbols, and Accents

Before comparison:

- Diacritical marks are removed: `José` becomes `Jose`.
- Standard spaces, non-breaking spaces, and wide spaces are normalized.
- For comparison only, periods, hyphens, and other non-letter characters are removed, and text is converted to lowercase.

Therefore, `J.P.`, `J-P`, and `J P` are all normalized to `jp`.


In [ ]:
def strip_accents(s):
    return "".join(
        c for c in unicodedata.normalize("NFKD", str(s))
        if not unicodedata.combining(c)
    )


def norm_spaces(s):
    return re.sub(
        r"\s+", " ",
        str(s).replace("\xa0", " ").replace("\u3000", " ")
    ).strip()


def compact(s):
    return re.sub(
        r"[^A-Za-z]", "",
        strip_accents(norm_spaces(s))
    ).lower()

examples = ["J.P.", "J-P", "J P", "  José\xa0Pérez  "]
pd.DataFrame({"original": examples, "normalized": [compact(x) for x in examples]})


,original,normalized
0,J.P.,jp
1,J-P,jp
2,J P,jp
3,José Pérez,joseperez


## Rule 2 — Clean Full Names and Split Hyphenated Components

- Parenthetical content is removed.
- Different hyphen characters are standardized.
- `space_tokens` preserves components separated by spaces.
- `components` also splits hyphenated components.

For example, `Chi-Fai Ng` contains two space-separated tokens, but it becomes three initial-generating components: `Chi`, `Fai`, and `Ng`.


In [ ]:
def clean_raw_name(name):
    s = strip_accents(norm_spaces(name))
    s = re.sub(r"\([^)]*\)", " ", s)
    s = s.replace("’", "'").replace("`", "'")
    s = re.sub(r"\s*[-–—]\s*", "-", s)
    return norm_spaces(s)


def space_tokens(name):
    s = clean_raw_name(name)
    out = []
    for tok in s.split():
        tok = re.sub(r"[^A-Za-z-]", "", tok).strip("-")
        if tok:
            out.append(tok)
    return out


def components(name):
    out = []
    for tok in space_tokens(name):
        for part in re.split(r"[-–—]", tok):
            part = re.sub(r"[^A-Za-z]", "", part)
            if part:
                out.append(part)
    return out

example_name = "Chi-Fai Ng (corresponding author)"
print("Clean name:", clean_raw_name(example_name))
print("Space-separated tokens:", space_tokens(example_name))
print("Components after hyphen splitting:", components(example_name))


Clean name: Chi-Fai Ng
Space-separated tokens: ['Chi-Fai', 'Ng']
Components after hyphen splitting: ['Chi', 'Fai', 'Ng']


## Helper Function — Store an Alias with a Specificity Score

For every generated alias, the algorithm stores:

- The author's index in the `full name` list.
- A `specificity` score.
- The name of the rule that generated the alias.

When the same alias is generated for the same author by multiple rules, the version with the highest specificity score is retained.


In [ ]:
def add_alias(alias_map, norm, idx, specificity, rule):
    norm = compact(norm)
    if len(norm) < 2:
        return

    previous = alias_map[norm].get(idx)
    item = (specificity, rule)

    if previous is None or item[0] > previous[0]:
        alias_map[norm][idx] = item


## Rule 3 — Test Alternative Name Orders

A full name may be stored in a different order from the order used to form its initials. Up to four orders are tested for every name:

1. Original order.
2. Move the first component to the end.
3. Move the last component to the beginning.
4. Reverse all components.

For example, `Qiu Tiantian` is also tested as `Tiantian Qiu`, which generates `TQ`.


In [ ]:
def ordered_name_variants(name):
    toks = space_tokens(name)
    variants = []

    def add(tokens, label):
        if len(tokens) >= 2:
            key = " ".join(tokens)
            if key not in {x[0] for x in variants}:
                variants.append((key, label))

    add(toks, "original_order")

    if len(toks) >= 2:
        add(toks[1:] + toks[:1], "move_first_to_end")
        add(toks[-1:] + toks[:-1], "move_last_to_front")
        add(list(reversed(toks)), "reverse_order")

    return variants

pd.DataFrame(ordered_name_variants("Qiu Tiantian"), columns=["name_variant", "rule"])


,name_variant,rule
0,Qiu Tiantian,original_order
1,Tiantian Qiu,move_first_to_end


## Basic Alias-Generation Rules

The next function combines several basic alias rules. Each rule is then demonstrated separately in the following cells.


In [ ]:
def add_code_style_aliases(alias_map, name_variant, idx, order_rule, bonus=0):
    st = space_tokens(name_variant)
    comps = components(name_variant)

    # Rule: initials of all name components
    if len(comps) >= 2:
        initials = "".join(token[0] for token in comps)
        add_alias(
            alias_map, initials, idx, len(initials) + bonus,
            f"initials_{order_rule}"
        )

        # Rule: first component + last component
        add_alias(
            alias_map, comps[0][0] + comps[-1][0], idx, 2 + bonus,
            f"first_last_initials_{order_rule}"
        )

        # Special three-component rule: first + last + middle
        if len(comps) == 3:
            add_alias(
                alias_map,
                comps[0][0] + comps[-1][0] + comps[1][0],
                idx,
                3 + bonus,
                f"three_token_special_{order_rule}"
            )

    if len(st) >= 2:
        first, last = st[0], st[-1]
        first_clean = re.sub(r"[^A-Za-z]", "", first)
        last_clean = re.sub(r"[^A-Za-z]", "", last)

        if first_clean and last_clean:
            # Rule: first initial + full surname
            add_alias(
                alias_map,
                first_clean[0] + last_clean,
                idx,
                len(last_clean) + 5 + bonus,
                f"initial_surname_{order_rule}"
            )

            # Rule: first initial + surname prefix
            for n in range(2, min(6, len(last_clean)) + 1):
                add_alias(
                    alias_map,
                    first_clean[0] + last_clean[:n],
                    idx,
                    n + 4 + bonus,
                    f"initial_surname_prefix_{order_rule}"
                )

            # Rule: full given name + first surname initial
            add_alias(
                alias_map,
                first_clean + last_clean[0],
                idx,
                len(first_clean) + 5 + bonus,
                f"first_name_last_initial_{order_rule}"
            )

        # Rule: initials of all given names + full surname
        if len(comps) >= 3:
            given = comps[:-1]
            surname = comps[-1]
            initials = "".join(x[0] for x in given)
            add_alias(
                alias_map,
                initials + surname,
                idx,
                len(initials) + len(surname) + 7 + bonus,
                f"multi_given_initials_surname_{order_rule}"
            )

        # Rule: surname particle, for example John van Dijk -> JVD
        if (
            len(st) >= 3
            and re.sub(r"[^A-Za-z]", "", st[-2]).lower() in PREFIX_TOKENS
        ):
            particle = re.sub(r"[^A-Za-z]", "", st[-2])
            if first_clean and particle and last_clean:
                add_alias(
                    alias_map,
                    first_clean[0] + particle[0] + last_clean[0],
                    idx,
                    8 + bonus,
                    f"particle_initials_{order_rule}"
                )

        # Rule: the full name itself as an alias
        full_alias = "".join(comps)
        add_alias(
            alias_map,
            full_alias,
            idx,
            len(full_alias) + 100 + bonus,
            f"full_name_{order_rule}"
        )


## Rule 4 — Initials of All Name Components

Take the first letter of every name component, including components created by splitting a hyphenated token.


In [ ]:
def aliases_from_basic_rules(name):
    alias_map = defaultdict(dict)
    add_code_style_aliases(alias_map, name, 0, "original_order")
    return pd.DataFrame([
        {"alias": alias, "specificity": value[0], "rule": value[1]}
        for alias, authors in alias_map.items()
        for _, value in authors.items()
    ]).sort_values(["rule", "alias"]).reset_index(drop=True)

basic_demo = aliases_from_basic_rules("Chi-Fai Ng")
basic_demo[basic_demo["rule"].str.startswith("initials_")]


,alias,specificity,rule
4,cfn,3,initials_original_order


## Rule 5 — First Letter of the First and Last Components

Middle-name initials may be omitted from an observed author alias.


In [ ]:
demo = aliases_from_basic_rules("Carl William Young")
demo[demo["rule"].str.startswith("first_last_initials_")]


,alias,specificity,rule
0,cy,2,first_last_initials_original_order


## Rule 6 — Special Three-Component Order

For names with exactly three components, the following order is also tested:

`first + last + middle`

Example: `Lee W Jones` → `LJW`.


In [ ]:
demo = aliases_from_basic_rules("Lee W Jones")
demo[demo["rule"].str.startswith("three_token_special_")]


,alias,specificity,rule
9,ljw,3,three_token_special_original_order


## Rule 7 — First Initial Plus Full Surname

The observed text `J. Smith` is normalized to `jsmith`, allowing it to match the corresponding generated alias.


In [ ]:
demo = aliases_from_basic_rules("John Smith")
demo[demo["rule"].str.startswith("initial_surname_original")]


,alias,specificity,rule
2,jsmith,10,initial_surname_original_order


## Rule 8 — First Initial Plus a Surname Prefix

Prefixes containing two to five letters from the surname are generated.


In [ ]:
demo = aliases_from_basic_rules("John Smith")
demo[demo["rule"].str.startswith("initial_surname_prefix_")]


,alias,specificity,rule
3,jsm,6,initial_surname_prefix_original_order
4,jsmi,7,initial_surname_prefix_original_order
5,jsmit,8,initial_surname_prefix_original_order


## Rule 9 — Full Given Name Plus the First Surname Initial

This rule detects forms such as `John S.`.


In [ ]:
demo = aliases_from_basic_rules("John Smith")
demo[demo["rule"].str.startswith("first_name_last_initial_")]


,alias,specificity,rule
0,johns,9,first_name_last_initial_original_order


## Rule 10 — Initials of All Given Names Plus the Full Surname

This rule is important for mixed aliases in which one part consists of initials and the other part contains a surname.


In [ ]:
for name in ["Chi-Fai Ng", "Peter Ka-Fung Chiu"]:
    demo = aliases_from_basic_rules(name)
    display(demo[demo["rule"].str.startswith("multi_given_initials_surname_")])


,alias,specificity,rule
5,cfng,11,multi_given_initials_surname_original_order


,alias,specificity,rule
7,pkfchiu,14,multi_given_initials_surname_original_order


## Rule 11 — Surname Particles

When the component immediately before the surname is a recognized particle, an alias containing that particle is generated.


In [ ]:
for name in ["John van Dijk", "Anna de Vries"]:
    demo = aliases_from_basic_rules(name)
    display(demo[demo["rule"].str.startswith("particle_initials_")])


,alias,specificity,rule
7,jvd,8,particle_initials_original_order


,alias,specificity,rule
8,adv,8,particle_initials_original_order


## Rule 12 — The Full Name as an Alias

When a full name appears directly inside the `authors` column, it can serve as a direct author identifier.

`Alexander Kawrykow` is normalized to `alexanderkawrykow` and receives a particularly high specificity score.


In [ ]:
demo = aliases_from_basic_rules("Alexander Kawrykow")
demo[demo["rule"].str.startswith("full_name_")]


,alias,specificity,rule
1,alexanderkawrykow,117,full_name_original_order


## Rule 13 — Permutations and Omission of Middle Components

For names containing two to five components, combinations and orders of two to four initials are generated:

- Permutations of the initials.
- Subsets that allow a middle name to be omitted from the observed alias.

For example, `Michel Nathan James` may generate `MNJ`, `MJN`, `NMJ`, `NJ`, `MJ`, and additional variants.

This is an expansive rule. Any collisions it creates are not accepted automatically; they are passed to the collision-resolution stage.


In [ ]:
def build_alias_map(names):
    alias_map = defaultdict(dict)

    for idx, name in enumerate(names):
        comps = components(name)

        # Apply all basic rules to every tested name order
        for variant_index, (variant, label) in enumerate(ordered_name_variants(name)):
            add_code_style_aliases(
                alias_map,
                variant,
                idx,
                label,
                bonus=(3 if variant_index else 0)
            )

        # Generate permutations and subsets of initials
        if 2 <= len(comps) <= 5:
            chars = [component[0] for component in comps]
            base = "".join(chars)
            max_k = min(4, len(chars))
            generated = set()

            for k in range(2, max_k + 1):
                for chosen in itertools.combinations(range(len(chars)), k):
                    values = [chars[j] for j in chosen]
                    for permutation in set(itertools.permutations(values)):
                        alias = "".join(permutation)
                        if alias in generated:
                            continue
                        generated.add(alias)

                        if alias == base:
                            rule = "initials_original"
                        elif k < len(chars):
                            rule = "initials_subset_permutation"
                        else:
                            rule = "initials_permutation"

                        add_alias(alias_map, alias, idx, len(alias), rule)

        # Two-component names with a compound given name or East Asian name order
        st = space_tokens(name)
        if len(st) == 2:
            first = re.sub(r"[^A-Za-z]", "", st[0])
            second = re.sub(r"[^A-Za-z]", "", st[1])

            for surname, given, label in [
                (first, second, "first_token_surname"),
                (second, first, "last_token_surname"),
            ]:
                if not surname or not given:
                    continue

                # First given-name letter + each plausible later letter + surname initial
                for j in range(1, len(given)):
                    alias = given[0] + given[j] + surname[0]
                    add_alias(
                        alias_map, alias, idx, 4,
                        f"compound_given_{label}"
                    )
                    add_alias(
                        alias_map,
                        surname[0] + given[0] + given[j],
                        idx,
                        4,
                        f"compound_given_reverse_{label}"
                    )

    return alias_map


def aliases_table(name, rule_contains=None):
    alias_map = build_alias_map([name])
    rows = []
    for alias, author_map in alias_map.items():
        for author_index, (specificity, rule) in author_map.items():
            rows.append({
                "alias": alias,
                "specificity": specificity,
                "rule": rule,
            })
    frame = pd.DataFrame(rows).sort_values(["rule", "alias"]).reset_index(drop=True)
    if rule_contains:
        frame = frame[frame["rule"].str.contains(rule_contains, regex=False)]
    return frame

aliases_table("Michel Nathan James", "permutation").head(30)


,alias,specificity,rule
34,mn,2,initials_subset_permutation
35,nj,2,initials_subset_permutation


## Rule 14 — Two-Component Names with Compound Given Names or East Asian Name Order

For a two-component name, both components are tested in both the `surname` and `given name` roles.

In addition to the first letter of the given name, later letters in the given name are tested as possible initials of an additional syllable or joined name component.

For example, `Qiu Tiantian` can generate three-letter aliases such as `TTQ` or `QTT` because another `T` occurs later in `Tiantian`.

> This is a heuristic. It does not perform linguistic syllable segmentation; it tests plausible later letters.


In [ ]:
compound_demo = aliases_table("Qiu Tiantian", "compound_given")
compound_demo.head(30)


,alias,specificity,rule
0,taq,4,compound_given_first_token_surname
1,tiq,4,compound_given_first_token_surname
2,tnq,4,compound_given_first_token_surname
3,ttq,4,compound_given_first_token_surname
4,qit,4,compound_given_last_token_surname
5,qut,4,compound_given_last_token_surname
6,qta,4,compound_given_reverse_first_token_surname
7,qtn,4,compound_given_reverse_first_token_surname
8,qtt,4,compound_given_reverse_first_token_surname
9,tqu,4,compound_given_reverse_last_token_surname


## Rule 15 — Detect Author Mentions in the `authors` Column

The algorithm scans every group in the `authors` column and detects:

1. Multi-token aliases such as `A Kawrykow`.
2. Initials containing periods, such as `J.P.`.
3. Compact initials such as `GR`.
4. Mixed aliases such as `CFNg`.
5. An initial or initials followed by a surname, even when the complete expression has not yet been found in the alias dictionary.

For multi-token aliases, the longest possible match is tested first, up to five tokens.


In [ ]:
def parse_mentions(author_text, alias_map):
    s = strip_accents(norm_spaces(author_text))
    s = re.sub(r"\([^)]*\)", " ", s)
    s = DECL_RE.sub(" ", s)
    tokens = TOKEN_RE.findall(s)

    out = []
    i = 0

    while i < len(tokens):
        # Prefer the longest matching multi-token alias
        matched = False
        for n in range(min(5, len(tokens) - i), 1, -1):
            sequence = tokens[i:i + n]
            letters = ["".join(re.findall(r"[A-Za-z]", x)) for x in sequence]
            has_lower = any(any(char.islower() for char in x) for x in letters)

            if not has_lower:
                continue

            normalized = compact(" ".join(sequence))
            if normalized in alias_map:
                out.append({
                    "surface": " ".join(sequence),
                    "norm": normalized,
                    "token_start": i,
                    "token_count": n,
                })
                i += n
                matched = True
                break

        if matched:
            continue

        token = tokens[i]
        letters = "".join(re.findall(r"[A-Za-z]", token))
        lower = letters.lower()

        # Accept a common-word alias only when every letter is uppercase
        if (
            not letters
            or ((lower in STOP_WORDS) and not letters.isupper())
            or (letters.islower() and len(letters) > 1)
        ):
            i += 1
            continue

        # Initial or initials followed by a surname
        initialish = (
            len(letters) <= 5
            and letters.isupper()
            and (len(letters) == 1 or "." in token)
        )

        if initialish and i + 1 < len(tokens):
            next_token = tokens[i + 1]
            next_letters = "".join(re.findall(r"[A-Za-z]", next_token))

            if (
                len(next_letters) >= 2
                and next_letters[0].isupper()
                and any(char.islower() for char in next_letters)
            ):
                surface = token + " " + next_token
                out.append({
                    "surface": surface,
                    "norm": compact(surface),
                    "token_start": i,
                    "token_count": 2,
                })
                i += 2
                continue

        # Compact initials or a mixed alias
        is_compact = letters.isupper() and len(letters) >= 2
        is_mixed_alias = (
            letters
            and letters[0].isupper()
            and any(char.islower() for char in letters[1:])
            and any(char.isupper() for char in letters[1:])
        )

        if is_compact or is_mixed_alias:
            out.append({
                "surface": token,
                "norm": compact(token),
                "token_start": i,
                "token_count": 1,
            })

        i += 1

    return out

names_demo = [
    "Alexander Kawrykow", "Gregory Rost", "Laura Smith",
    "Michael Brown", "James White"
]
alias_map_demo = build_alias_map(names_demo)
parse_mentions("A Kawrykow GR LS MB JW", alias_map_demo)


[{'surface': 'A Kawrykow',
  'norm': 'akawrykow',
  'token_start': 0,
  'token_count': 2},
 {'surface': 'GR', 'norm': 'gr', 'token_start': 2, 'token_count': 1},
 {'surface': 'LS', 'norm': 'ls', 'token_start': 3, 'token_count': 1},
 {'surface': 'MB', 'norm': 'mb', 'token_start': 4, 'token_count': 1},
 {'surface': 'JW', 'norm': 'jw', 'token_start': 5, 'token_count': 1}]

## Rule 16 — Common Words Are Accepted Only in UPPERCASE

Words such as `the`, `all`, `has`, and `and` may accidentally resemble author initials.

Therefore:

- `the` and `The` are rejected.
- `THE` and `T.H.E.` may be accepted.
- `has` is rejected, while `HAS` may be accepted.

This preserves genuine aliases without interpreting ordinary prose as an author list.


In [ ]:
uppercase_names = [
    "Thomas Henry Evans",  # THE
    "Helen Amy Stone",     # HAS
]
uppercase_map = build_alias_map(uppercase_names)

for text in ["the has", "The Has", "THE HAS", "T.H.E. H.A.S."]:
    print(text, "->", parse_mentions(text, uppercase_map))


the has -> []
The Has -> []
THE HAS -> [{'surface': 'THE', 'norm': 'the', 'token_start': 0, 'token_count': 1}, {'surface': 'HAS', 'norm': 'has', 'token_start': 1, 'token_count': 1}]
T.H.E. H.A.S. -> [{'surface': 'T.H.E.', 'norm': 'the', 'token_start': 0, 'token_count': 1}, {'surface': 'H.A.S.', 'norm': 'has', 'token_start': 1, 'token_count': 1}]


## Rule 17 — Remove General Author Declarations

Text beginning with `All authors`, `Both authors`, `The authors`, or `Author(s)` is removed through the end of the group.

The purpose is to avoid interpreting a sentence such as:

`All authors have read and approved the manuscript`

as a list of author initials.


In [ ]:
declaration_map = build_alias_map(["Thomas Henry Evans"])

for text in [
    "All authors have read and approved the manuscript",
    "THE",
    "The authors confirm that they meet the ICMJE criteria",
]:
    print(text, "->", parse_mentions(text, declaration_map))


All authors have read and approved the manuscript -> []
THE -> [{'surface': 'THE', 'norm': 'the', 'token_start': 0, 'token_count': 1}]
The authors confirm that they meet the ICMJE criteria -> []


## Rule 18 — Split Accidentally Concatenated Aliases

When an unknown compact sequence is found, the algorithm attempts to split it into known aliases.

For example, `PJJT` can be split into `PJ + JT` only when:

1. Every part exists in the alias dictionary for the same article.
2. Every part maps uniquely to one author.
3. The entire sequence can be segmented.
4. The number of resulting parts does not exceed four.


In [ ]:
def split_concatenated(norm, alias_map, max_parts=4):
    memo = {}

    def recurse(position, parts):
        key = (position, parts)
        if key in memo:
            return memo[key]
        if position == len(norm):
            return []
        if parts >= max_parts:
            return None

        best = None
        for end in range(len(norm), position + 1, -1):
            segment = norm[position:end]
            candidates = alias_map.get(segment)

            # Every segment must map uniquely
            if not candidates or len(candidates) != 1:
                continue

            rest = recurse(end, parts + 1)
            if rest is None:
                continue

            candidate = [segment] + rest
            if (
                best is None
                or (len(candidate), [-len(x) for x in candidate])
                < (len(best), [-len(x) for x in best])
            ):
                best = candidate

        memo[key] = best
        return best

    answer = recurse(0, 0)
    return answer if answer and len(answer) >= 2 else None

concat_map = build_alias_map(["Peter Jones", "James Taylor"])
print("PJJT ->", split_concatenated("pjjt", concat_map))


PJJT -> ['pj', 'jt']


## Rule 19 — Resolve Collisions Between Full Names

A single observed initial may match more than one distinct full name. The resolver first prefers a unique longer observed alias and then prefers a uniquely highest-specificity rule. If no safe winner exists, the initial remains `ambiguous`.

Duplicate occurrences of the **same full name** are not treated as different people in the output. They are collapsed into one name entity and their original list indices are retained together.


In [ ]:
def normalize_full_name_key(name):
    """Return a stable key used to collapse repeated occurrences of the same name."""
    return compact(clean_raw_name(name))


def build_name_entities(full_names):
    """
    Collapse identical normalized full names into one entity.

    Example:
        ["John Smith", "Jane Stone", "John Smith"]
    becomes:
        John Smith -> indices [0, 2]
        Jane Stone -> indices [1]
    """
    entities = []
    key_to_entity = {}

    for original_index, raw_name in enumerate(full_names):
        clean_name = clean_raw_name(raw_name)
        key = normalize_full_name_key(clean_name)

        if not key:
            continue

        if key not in key_to_entity:
            entity_id = len(entities)
            key_to_entity[key] = entity_id
            entities.append({
                "entity_id": entity_id,
                "full_name": clean_name,
                "full_name_indices": [original_index],
            })
        else:
            entity_id = key_to_entity[key]
            entities[entity_id]["full_name_indices"].append(original_index)

    return entities


def add_unique_initial(container, order, mention, source_item_index):
    """Add one normalized initial to an article-level unique set."""
    norm = mention["norm"]

    if norm not in container:
        container[norm] = {
            "norm": norm,
            "observed_forms": [],
            "observed_form_set": set(),
            "source_item_indices": [],
            "source_item_index_set": set(),
            "split_from": [],
            "split_from_set": set(),
        }
        order.append(norm)

    current = container[norm]
    surface = mention["surface"]

    if surface not in current["observed_form_set"]:
        current["observed_form_set"].add(surface)
        current["observed_forms"].append(surface)

    if source_item_index not in current["source_item_index_set"]:
        current["source_item_index_set"].add(source_item_index)
        current["source_item_indices"].append(source_item_index)

    split_from = mention.get("split_from", "")
    if split_from and split_from not in current["split_from_set"]:
        current["split_from_set"].add(split_from)
        current["split_from"].append(split_from)


def extract_unique_initials(author_items, alias_map):
    """
    Flatten every list item in `authors` into one unique set of normalized initials.

    Repeated initials are stored once, even when they occur in several list items.
    """
    unique = {}
    order = []

    for source_item_index, author_text in enumerate(author_items):
        mentions = parse_mentions(author_text, alias_map)

        for mention in mentions:
            expanded = []

            if mention["norm"] not in alias_map:
                segments = split_concatenated(mention["norm"], alias_map)
                if segments:
                    for segment in segments:
                        split_mention = dict(mention)
                        split_mention["surface"] = segment.upper()
                        split_mention["norm"] = segment
                        split_mention["split_from"] = mention["surface"]
                        expanded.append(split_mention)
                else:
                    mention = dict(mention)
                    mention["split_from"] = ""
                    expanded.append(mention)
            else:
                mention = dict(mention)
                mention["split_from"] = ""
                expanded.append(mention)

            for expanded_mention in expanded:
                add_unique_initial(
                    unique,
                    order,
                    expanded_mention,
                    source_item_index,
                )

    return [unique[norm] for norm in order]


def resolve_unique_initials(author_items, full_names):
    """
    Resolve one unique set of article initials against unique full-name entities.

    Returns:
        results: one result per unique normalized initial
        entities: one entity per unique full name, including all original indices
        alias_map: generated aliases for the unique full-name entities
    """
    entities = build_name_entities(full_names)
    entity_names = [entity["full_name"] for entity in entities]
    alias_map = build_alias_map(entity_names)
    unique_initials = extract_unique_initials(author_items, alias_map)

    # Collect uniquely identifiable longer/specific observed aliases.
    unique_specific = defaultdict(list)

    for mention in unique_initials:
        candidates = alias_map.get(mention["norm"], {})
        chosen = None

        if len(candidates) == 1:
            entity_id, (specificity, rule) = next(iter(candidates.items()))
            chosen = (entity_id, specificity)
        elif candidates:
            max_specificity = max(value[0] for value in candidates.values())
            top = [
                (entity_id, value[0])
                for entity_id, value in candidates.items()
                if value[0] == max_specificity
            ]
            if len(top) == 1:
                chosen = top[0]

        if chosen:
            unique_specific[chosen[0]].append((mention["norm"], chosen[1]))

    results = []

    for mention in unique_initials:
        candidates = alias_map.get(mention["norm"], {})
        result = {
            "initials": mention["observed_forms"][0],
            "observed_forms": list(mention["observed_forms"]),
            "normalized_initials": mention["norm"],
            "source_item_indices": list(mention["source_item_indices"]),
            "split_from": list(mention["split_from"]),
            "status": "",
            "entity_id": None,
            "full_name": "",
            "full_name_indices": [],
            "candidate_entity_ids": [],
            "candidate_full_names": [],
            "candidate_indices": [],
            "rule": "",
        }

        if not candidates:
            result["status"] = "unmapped"
            results.append(result)
            continue

        if len(candidates) == 1:
            entity_id, (specificity, rule) = next(iter(candidates.items()))
            entity = entities[entity_id]
            result.update(
                status="mapped",
                entity_id=entity_id,
                full_name=entity["full_name"],
                full_name_indices=list(entity["full_name_indices"]),
                candidate_entity_ids=[entity_id],
                candidate_full_names=[entity["full_name"]],
                candidate_indices=[list(entity["full_name_indices"])],
                rule=rule,
            )
            results.append(result)
            continue

        filtered = []
        for entity_id, (specificity, rule) in candidates.items():
            has_longer_observed_alias = any(
                len(alias) > len(mention["norm"])
                for alias, _ in unique_specific.get(entity_id, [])
            )
            if not has_longer_observed_alias:
                filtered.append((entity_id, specificity, rule))

        if filtered:
            max_specificity = max(item[1] for item in filtered)
            top = [item for item in filtered if item[1] == max_specificity]
        else:
            top = []

        if len(top) == 1:
            entity_id, specificity, rule = top[0]
            entity = entities[entity_id]
            candidate_ids = list(candidates)
            result.update(
                status="mapped_resolved",
                entity_id=entity_id,
                full_name=entity["full_name"],
                full_name_indices=list(entity["full_name_indices"]),
                candidate_entity_ids=candidate_ids,
                candidate_full_names=[entities[j]["full_name"] for j in candidate_ids],
                candidate_indices=[list(entities[j]["full_name_indices"]) for j in candidate_ids],
                rule=rule,
            )
        else:
            candidate_ids = [item[0] for item in top] if top else list(candidates)
            result.update(
                status="ambiguous",
                candidate_entity_ids=candidate_ids,
                candidate_full_names=[entities[j]["full_name"] for j in candidate_ids],
                candidate_indices=[list(entities[j]["full_name_indices"]) for j in candidate_ids],
            )

        results.append(result)

    return results, entities, alias_map


# Collision and repeated-name example.
collision_full_names = ["John Smith", "Jane Stone", "John Smith"]
collision_authors = ["JS J Smith"]
collision_results, collision_entities, _ = resolve_unique_initials(
    collision_authors,
    collision_full_names,
)

print("Unique full-name entities:")
display(pd.DataFrame(collision_entities))
print("Unique initials resolution:")
display(pd.DataFrame(collision_results))


Unique full-name entities:


,entity_id,full_name,full_name_indices
0,0,John Smith,"[0, 2]"
1,1,Jane Stone,[1]


Unique initials resolution:


,initials,observed_forms,normalized_initials,source_item_indices,split_from,status,entity_id,full_name,full_name_indices,candidate_entity_ids,candidate_full_names,candidate_indices,rule
0,JS,[JS],js,[0],[],mapped_resolved,1,Jane Stone,[1],"[0, 1]","[John Smith, Jane Stone]","[[0, 2], [1]]",initials_original_order
1,J Smith,[J Smith],jsmith,[0],[],mapped,0,John Smith,"[0, 2]",[0],[John Smith],"[[0, 2]]",initial_surname_original_order


## 2. Inspect One Article
The next function returns two tables:

1. **Full-name table:** one row per unique full name, including all zero-based indices and all mapped initials.
2. **Unique-initials table:** one row per unique normalized initial, including status, selected name, candidate names, and rule.

No author-group dictionary is produced.


In [ ]:
def append_unique(target, value):
    if value not in target:
        target.append(value)


def pipe_join(values):
    return " | ".join(str(value) for value in values)


def format_nested_indices(index_lists):
    return " | ".join(
        ",".join(str(index) for index in indices)
        for indices in index_lists
    )


def build_full_name_mapping_rows(
    source_row_index,
    title,
    year,
    entities,
    results,
):
    """Create one readable output row per unique full name."""
    state = {
        entity["entity_id"]: {
            "mapped_initials": [],
            "mapped_normalized_initials": [],
            "ambiguous_initials": [],
            "ambiguous_normalized_initials": [],
            "rules": [],
        }
        for entity in entities
    }

    for result in results:
        display_initial = result["initials"]
        normalized = result["normalized_initials"]

        if result["status"] in {"mapped", "mapped_resolved"}:
            entity_state = state[result["entity_id"]]
            append_unique(entity_state["mapped_initials"], display_initial)
            append_unique(entity_state["mapped_normalized_initials"], normalized)
            if result.get("rule"):
                append_unique(entity_state["rules"], result["rule"])

        elif result["status"] == "ambiguous":
            for entity_id in result["candidate_entity_ids"]:
                entity_state = state[entity_id]
                append_unique(entity_state["ambiguous_initials"], display_initial)
                append_unique(entity_state["ambiguous_normalized_initials"], normalized)

    rows = []

    for entity in sorted(entities, key=lambda item: item["full_name_indices"][0]):
        entity_state = state[entity["entity_id"]]
        has_mapped = bool(entity_state["mapped_initials"])
        has_ambiguous = bool(entity_state["ambiguous_initials"])

        if has_mapped and has_ambiguous:
            mapping_status = "mapped_with_ambiguous_initials"
        elif has_mapped:
            mapping_status = "mapped"
        elif has_ambiguous:
            mapping_status = "ambiguous_only"
        else:
            mapping_status = "unmapped"

        indices = entity["full_name_indices"]
        rows.append({
            "source_row_index": source_row_index,
            "title": title,
            "year": year,
            "full_name": entity["full_name"],
            "full_name_indices": pipe_join(indices),
            "first_full_name_index": indices[0],
            "number_of_full_name_indices": len(indices),
            "initials": pipe_join(entity_state["mapped_initials"]),
            "normalized_initials": pipe_join(entity_state["mapped_normalized_initials"]),
            "number_of_initials": len(entity_state["mapped_normalized_initials"]),
            "ambiguous_initials": pipe_join(entity_state["ambiguous_initials"]),
            "mapping_status": mapping_status,
            "rules": pipe_join(entity_state["rules"]),
        })

    return rows


def build_unique_initial_rows(source_row_index, title, year, results):
    """Create one readable row per unique normalized initial."""
    rows = []

    for result in results:
        rows.append({
            "source_row_index": source_row_index,
            "title": title,
            "year": year,
            "initials": result["initials"],
            "observed_forms": pipe_join(result["observed_forms"]),
            "normalized_initials": result["normalized_initials"],
            "status": result["status"],
            "full_name": result["full_name"],
            "full_name_indices": pipe_join(result["full_name_indices"]),
            "candidate_full_names": pipe_join(result["candidate_full_names"]),
            "candidate_indices": format_nested_indices(result["candidate_indices"]),
            "rule": result["rule"],
            "split_from": pipe_join(result["split_from"]),
        })

    return rows


def inspect_one_article(author_items, full_names):
    results, entities, alias_map = resolve_unique_initials(
        author_items,
        full_names,
    )

    full_name_table = pd.DataFrame(
        build_full_name_mapping_rows(
            source_row_index=0,
            title="Example article",
            year="",
            entities=entities,
            results=results,
        )
    )

    unique_initials_table = pd.DataFrame(
        build_unique_initial_rows(
            source_row_index=0,
            title="Example article",
            year="",
            results=results,
        )
    )

    return full_name_table, unique_initials_table, alias_map


example_authors = [
    "A Kawrykow GR LS MB JW",
    "A Kawrykow GR A Kam DK CL CW EZ",
    "MB JW",
]

example_full_names = [
    "Alexander Kawrykow",
    "Gregory Rost",
    "Laura Smith",
    "Michael Brown",
    "James White",
    "Amy Kam",
    "Daniel King",
    "Carol Lee",
    "Chris Wong",
    "Emily Zhang",
]

example_full_name_table, example_initials_table, _ = inspect_one_article(
    example_authors,
    example_full_names,
)

print("Full name to initials mapping:")
display(example_full_name_table)
print("Unique initials set and resolution:")
display(example_initials_table)


Full name to initials mapping:


,source_row_index,title,year,full_name,full_name_indices,first_full_name_index,number_of_full_name_indices,initials,normalized_initials,number_of_initials,ambiguous_initials,mapping_status,rules
0,0,Example article,,Alexander Kawrykow,0,0,1,A Kawrykow,akawrykow,1,,mapped,initial_surname_original_order
1,0,Example article,,Gregory Rost,1,1,1,GR,gr,1,,mapped,initials_original_order
2,0,Example article,,Laura Smith,2,2,1,LS,ls,1,,mapped,initials_original_order
3,0,Example article,,Michael Brown,3,3,1,MB,mb,1,,mapped,initials_original_order
4,0,Example article,,James White,4,4,1,JW,jw,1,,mapped,initials_original_order
5,0,Example article,,Amy Kam,5,5,1,A Kam,akam,1,,mapped,initial_surname_original_order
6,0,Example article,,Daniel King,6,6,1,DK,dk,1,,mapped,initials_original_order
7,0,Example article,,Carol Lee,7,7,1,CL,cl,1,,mapped,initials_original_order
8,0,Example article,,Chris Wong,8,8,1,CW,cw,1,,mapped,initials_original_order
9,0,Example article,,Emily Zhang,9,9,1,EZ,ez,1,,mapped,initials_original_order


Unique initials set and resolution:


,source_row_index,title,year,initials,observed_forms,normalized_initials,status,full_name,full_name_indices,candidate_full_names,candidate_indices,rule,split_from
0,0,Example article,,A Kawrykow,A Kawrykow,akawrykow,mapped,Alexander Kawrykow,0,Alexander Kawrykow,0,initial_surname_original_order,
1,0,Example article,,GR,GR,gr,mapped,Gregory Rost,1,Gregory Rost,1,initials_original_order,
2,0,Example article,,LS,LS,ls,mapped,Laura Smith,2,Laura Smith,2,initials_original_order,
3,0,Example article,,MB,MB,mb,mapped,Michael Brown,3,Michael Brown,3,initials_original_order,
4,0,Example article,,JW,JW,jw,mapped,James White,4,James White,4,initials_original_order,
5,0,Example article,,A Kam,A Kam,akam,mapped,Amy Kam,5,Amy Kam,5,initial_surname_original_order,
6,0,Example article,,DK,DK,dk,mapped,Daniel King,6,Daniel King,6,initials_original_order,
7,0,Example article,,CL,CL,cl,mapped,Carol Lee,7,Carol Lee,7,initials_original_order,
8,0,Example article,,CW,CW,cw,mapped,Chris Wong,8,Chris Wong,8,initials_original_order,
9,0,Example article,,EZ,EZ,ez,mapped,Emily Zhang,9,Emily Zhang,9,initials_original_order,


## 3. Dataset File Helper Functions

The `authors` and `full name` columns are stored in the CSV as textual Python lists. The notebook therefore uses `ast.literal_eval` rather than `eval`.

The index stored in `full_name_indices` is the original **zero-based Python index** inside the `full name` list. Repeated identical names receive several indices in the same CSV row.


In [ ]:
def parse_list_cell(value):
    """Parse a CSV cell containing a Python-style list."""
    if isinstance(value, list):
        return value

    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []

    text = str(value).strip()
    if not text:
        return []

    parsed = ast.literal_eval(text)
    if not isinstance(parsed, list):
        raise ValueError(
            f"Expected list, received {type(parsed).__name__}: {text[:200]}"
        )

    return [str(item) for item in parsed]


def status_for_unique_results(results):
    total = len(results)
    success = sum(
        result["status"] in {"mapped", "mapped_resolved"}
        for result in results
    )

    if total == 0:
        return "no_initials"
    if success == total:
        return "fully_mapped"
    if success == 0:
        return "none_mapped"
    return "partially_mapped"


## 4. CSV Outputs

The main function creates four CSV files:

- `full_name_initials_mapping.csv` — **one row per unique full name** with its original list index or indices and all mapped initials.
- `unique_initials_mapping.csv` — one row per unique initial found in the entire `authors` list of an article.
- `article_mapping_summary.csv` — one compact summary row per article.
- `initials_mapping_failures.csv` — only ambiguous or unmapped unique initials.

No group-level JSON dictionary is written. All multi-value fields use a readable ` | ` separator.


In [ ]:
# Preview the exact columns used in the main full-name CSV.
FULL_NAME_OUTPUT_COLUMNS = [
    "source_row_index",
    "title",
    "year",
    "full_name",
    "full_name_indices",
    "first_full_name_index",
    "number_of_full_name_indices",
    "initials",
    "normalized_initials",
    "number_of_initials",
    "ambiguous_initials",
    "mapping_status",
    "rules",
]

pd.DataFrame(columns=FULL_NAME_OUTPUT_COLUMNS)


,source_row_index,title,year,full_name,full_name_indices,first_full_name_index,number_of_full_name_indices,initials,normalized_initials,number_of_initials,ambiguous_initials,mapping_status,rules


## 5. Main CSV Processing Function

The function reads the source CSV in chunks, flattens all values from `authors` into one unique initials set per article, resolves each unique initial once, and then writes one readable row per full name.

The alias-generation and collision-resolution rules above are unchanged. Only the dataset aggregation and CSV layout are different.


In [ ]:
def run_author_mapping(
    input_path,
    output_dir,
    authors_col="authors",
    full_name_col="full name",
    title_col="title",
    year_col="year",
    chunksize=2000,
    max_rows=None,
    progress_every=5000,
    create_zip=True,
):
    """
    Process a CSV and create readable full-name-to-initials mappings.

    Important behavior:
    - Every `authors` list is flattened into one unique initials set.
    - Every unique normalized initial is resolved only once per article.
    - The main output contains one row per unique full name.
    - Duplicate identical full names keep all original zero-based indices.
    """
    started = time.time()
    input_path = str(input_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    full_name_path = output_dir / "full_name_initials_mapping.csv"
    initials_path = output_dir / "unique_initials_mapping.csv"
    article_path = output_dir / "article_mapping_summary.csv"
    failures_path = output_dir / "initials_mapping_failures.csv"
    summary_path = output_dir / "mapping_summary.json"

    full_name_fields = FULL_NAME_OUTPUT_COLUMNS

    initials_fields = [
        "source_row_index", "title", "year", "initials", "observed_forms",
        "normalized_initials", "status", "full_name", "full_name_indices",
        "candidate_full_names", "candidate_indices", "rule", "split_from",
    ]

    article_fields = [
        "source_row_index", "title", "year", "authors", "full_name",
        "unique_initials", "normalized_unique_initials",
        "unique_initials_total", "mapped_unique_initials",
        "resolved_unique_initials", "ambiguous_unique_initials",
        "unmapped_unique_initials", "mapping_status",
        "full_name_entries_total", "unique_full_names_total",
        "duplicate_full_name_entries", "full_names_with_initials",
        "full_names_without_initials",
    ]

    failure_fields = [
        "source_row_index", "title", "year", "initials", "observed_forms",
        "normalized_initials", "status", "candidate_full_names",
        "candidate_indices", "split_from",
    ]

    counts = Counter()
    row_status_counts = Counter()
    name_status_counts = Counter()
    rule_counts = Counter()
    unmapped_initial_counts = Counter()
    ambiguous_initial_counts = Counter()
    distinct_initials = set()
    failure_examples = []
    ambiguous_examples = []

    usecols = [title_col, year_col, authors_col, full_name_col]

    reader = pd.read_csv(
        input_path,
        usecols=usecols,
        chunksize=chunksize,
        encoding="utf-8-sig",
        low_memory=False,
    )

    processed = 0
    next_progress = progress_every

    with (
        open(full_name_path, "w", newline="", encoding="utf-8-sig") as full_name_file,
        open(initials_path, "w", newline="", encoding="utf-8-sig") as initials_file,
        open(article_path, "w", newline="", encoding="utf-8-sig") as article_file,
        open(failures_path, "w", newline="", encoding="utf-8-sig") as failure_file,
    ):
        full_name_writer = csv.DictWriter(full_name_file, fieldnames=full_name_fields)
        initials_writer = csv.DictWriter(initials_file, fieldnames=initials_fields)
        article_writer = csv.DictWriter(article_file, fieldnames=article_fields)
        failure_writer = csv.DictWriter(failure_file, fieldnames=failure_fields)

        full_name_writer.writeheader()
        initials_writer.writeheader()
        article_writer.writeheader()
        failure_writer.writeheader()

        stop = False

        for chunk in reader:
            if max_rows is not None:
                remaining = int(max_rows) - processed
                if remaining <= 0:
                    break
                if len(chunk) > remaining:
                    chunk = chunk.iloc[:remaining]
                    stop = True

            for _, row in chunk.iterrows():
                source_row_index = processed
                title = row[title_col]
                year = row[year_col]
                original_authors_cell = row[authors_col]
                original_full_name_cell = row[full_name_col]

                author_items = parse_list_cell(original_authors_cell)
                full_names = parse_list_cell(original_full_name_cell)

                results, entities, _ = resolve_unique_initials(
                    author_items,
                    full_names,
                )

                unique_initial_rows = build_unique_initial_rows(
                    source_row_index,
                    title,
                    year,
                    results,
                )
                initials_writer.writerows(unique_initial_rows)

                full_name_rows = build_full_name_mapping_rows(
                    source_row_index,
                    title,
                    year,
                    entities,
                    results,
                )
                full_name_writer.writerows(full_name_rows)

                mapped_count = sum(r["status"] == "mapped" for r in results)
                resolved_count = sum(r["status"] == "mapped_resolved" for r in results)
                ambiguous_count = sum(r["status"] == "ambiguous" for r in results)
                unmapped_count = sum(r["status"] == "unmapped" for r in results)
                total_unique_initials = len(results)
                successful_count = mapped_count + resolved_count
                mapping_status = status_for_unique_results(results)
                row_status_counts[mapping_status] += 1

                full_names_with_initials = sum(
                    row_data["number_of_initials"] > 0
                    for row_data in full_name_rows
                )
                full_names_without_initials = len(full_name_rows) - full_names_with_initials
                duplicate_full_name_entries = len(full_names) - len(entities)

                for row_data in full_name_rows:
                    name_status_counts[row_data["mapping_status"]] += 1

                article_writer.writerow({
                    "source_row_index": source_row_index,
                    "title": title,
                    "year": year,
                    "authors": original_authors_cell,
                    "full_name": original_full_name_cell,
                    "unique_initials": pipe_join(r["initials"] for r in results),
                    "normalized_unique_initials": pipe_join(
                        r["normalized_initials"] for r in results
                    ),
                    "unique_initials_total": total_unique_initials,
                    "mapped_unique_initials": mapped_count,
                    "resolved_unique_initials": resolved_count,
                    "ambiguous_unique_initials": ambiguous_count,
                    "unmapped_unique_initials": unmapped_count,
                    "mapping_status": mapping_status,
                    "full_name_entries_total": len(full_names),
                    "unique_full_names_total": len(entities),
                    "duplicate_full_name_entries": duplicate_full_name_entries,
                    "full_names_with_initials": full_names_with_initials,
                    "full_names_without_initials": full_names_without_initials,
                })

                counts["rows_total"] += 1
                counts["authors_list_items_total"] += len(author_items)
                counts["full_name_entries_total"] += len(full_names)
                counts["unique_full_names_total"] += len(entities)
                counts["duplicate_full_name_entries"] += duplicate_full_name_entries
                counts["unique_initials_total"] += total_unique_initials
                counts["unique_initials_mapped"] += mapped_count
                counts["unique_initials_mapped_resolved"] += resolved_count
                counts["unique_initials_ambiguous"] += ambiguous_count
                counts["unique_initials_unmapped"] += unmapped_count
                counts["full_name_rows_total"] += len(full_name_rows)
                counts["full_names_with_initials"] += full_names_with_initials
                counts["full_names_without_initials"] += full_names_without_initials
                counts["names_with_multiple_indices"] += sum(
                    row_data["number_of_full_name_indices"] > 1
                    for row_data in full_name_rows
                )
                counts["names_with_multiple_initials"] += sum(
                    row_data["number_of_initials"] > 1
                    for row_data in full_name_rows
                )

                for result, output_row in zip(results, unique_initial_rows):
                    distinct_initials.add(result["normalized_initials"])

                    if result["status"] in {"mapped", "mapped_resolved"}:
                        rule_counts[result.get("rule", "")] += 1
                    elif result["status"] == "unmapped":
                        unmapped_initial_counts[result["normalized_initials"]] += 1
                    elif result["status"] == "ambiguous":
                        ambiguous_initial_counts[result["normalized_initials"]] += 1

                    if result["status"] in {"unmapped", "ambiguous"}:
                        failure_row = {
                            key: output_row[key]
                            for key in failure_fields
                        }
                        failure_writer.writerow(failure_row)

                        if result["status"] == "unmapped" and len(failure_examples) < 500:
                            failure_examples.append(dict(failure_row))

                        if result["status"] == "ambiguous" and len(ambiguous_examples) < 500:
                            ambiguous_examples.append(dict(failure_row))

                processed += 1

                if progress_every and processed >= next_progress:
                    elapsed = time.time() - started
                    total = counts["unique_initials_total"]
                    success = (
                        counts["unique_initials_mapped"]
                        + counts["unique_initials_mapped_resolved"]
                    )
                    rate = 100 * success / total if total else 0.0

                    print(
                        f"Rows: {processed:,} | Unique initials: {total:,} | "
                        f"Successful: {success:,} ({rate:.6f}%) | "
                        f"Ambiguous: {counts['unique_initials_ambiguous']:,} | "
                        f"Unmapped: {counts['unique_initials_unmapped']:,} | "
                        f"Elapsed: {elapsed / 60:.2f} min"
                    )
                    next_progress += progress_every

            del chunk
            gc.collect()

            if stop:
                break

    total_unique_initials = counts["unique_initials_total"]
    successful_unique_initials = (
        counts["unique_initials_mapped"]
        + counts["unique_initials_mapped_resolved"]
    )
    failed_unique_initials = (
        counts["unique_initials_ambiguous"]
        + counts["unique_initials_unmapped"]
    )

    summary = {
        "counts": dict(counts),
        "unique_initial_level": {
            "total_unique_initials": total_unique_initials,
            "successful_unique_initials": successful_unique_initials,
            "direct_mapped": counts["unique_initials_mapped"],
            "resolved_mapped": counts["unique_initials_mapped_resolved"],
            "ambiguous": counts["unique_initials_ambiguous"],
            "unmapped": counts["unique_initials_unmapped"],
            "success_rate_percent": (
                100 * successful_unique_initials / total_unique_initials
                if total_unique_initials else 0.0
            ),
            "failure_rate_percent": (
                100 * failed_unique_initials / total_unique_initials
                if total_unique_initials else 0.0
            ),
        },
        "row_status": dict(row_status_counts),
        "full_name_status": dict(name_status_counts),
        "rule_counts": dict(rule_counts),
        "top_unmapped_initials": [
            [key, value]
            for key, value in unmapped_initial_counts.most_common(100)
        ],
        "top_ambiguous_initials": [
            [key, value]
            for key, value in ambiguous_initial_counts.most_common(100)
        ],
        "distinct_normalized_initials": len(distinct_initials),
        "runtime_seconds": time.time() - started,
        "method_notes": [
            "All values in the authors list were flattened into one unique initials set per article.",
            "Each normalized initial was resolved only once per article.",
            "The main CSV contains one row per unique full name.",
            "Repeated identical full names keep all original zero-based indices in one row.",
            "Multiple mapped initials for one full name are stored in one readable pipe-separated field.",
            "Ambiguous and unmapped unique initials are exported separately.",
        ],
        "failure_examples": failure_examples,
        "ambiguous_examples": ambiguous_examples,
    }

    with open(summary_path, "w", encoding="utf-8") as summary_file:
        json.dump(summary, summary_file, ensure_ascii=False, indent=2)

    output_zip_path = None
    if create_zip:
        output_zip_path = shutil.make_archive(
            str(output_dir),
            "zip",
            root_dir=output_dir.parent,
            base_dir=output_dir.name,
        )

    print("\nDONE")
    print(json.dumps(summary["unique_initial_level"], ensure_ascii=False, indent=2))
    print(f"Full-name mapping: {full_name_path}")
    print(f"Unique initials: {initials_path}")
    print(f"Article summary: {article_path}")
    print(f"Failures: {failures_path}")
    print(f"Summary: {summary_path}")
    if output_zip_path:
        print(f"ZIP: {output_zip_path}")

    summary["output_files"] = {
        "full_name_mapping": str(full_name_path),
        "unique_initials": str(initials_path),
        "article_summary": str(article_path),
        "failures": str(failures_path),
        "summary": str(summary_path),
        "zip": output_zip_path,
    }

    return summary


## 6. Final Function for Processing a ZIP File

The function opens the ZIP archive, selects the requested CSV or the largest CSV, extracts it, and calls the complete readable mapping function.


In [ ]:
def run_mapping_from_zip(
    zip_path,
    output_dir="/content/author_initials_mapping_results",
    csv_name=None,
    authors_col="authors",
    full_name_col="full name",
    title_col="title",
    year_col="year",
    chunksize=2000,
    max_rows=None,
    progress_every=5000,
    create_zip=True,
):
    zip_path = Path(zip_path)
    output_dir = Path(output_dir)

    if not zip_path.exists():
        raise FileNotFoundError(f"ZIP file was not found: {zip_path}")

    if zip_path.suffix.lower() != ".zip":
        raise ValueError(f"Expected a .zip file, received: {zip_path.name}")

    cache_dir = output_dir.parent / f"{output_dir.name}_input_cache"
    cache_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as archive:
        csv_members = [
            info for info in archive.infolist()
            if not info.is_dir() and info.filename.lower().endswith(".csv")
        ]

        if not csv_members:
            raise ValueError("No CSV file was found inside the ZIP.")

        if csv_name is not None:
            matches = [
                info for info in csv_members
                if info.filename == csv_name
                or Path(info.filename).name == csv_name
            ]
            if not matches:
                available = [info.filename for info in csv_members]
                raise ValueError(
                    f"CSV '{csv_name}' was not found. Available CSV files: {available}"
                )
            selected = matches[0]
        elif len(csv_members) == 1:
            selected = csv_members[0]
        else:
            selected = max(csv_members, key=lambda info: info.file_size)
            print(
                "Several CSV files were found. "
                f"The largest file was selected: {selected.filename}"
            )

        extracted_path = Path(archive.extract(selected, path=cache_dir))

    print(f"Input ZIP: {zip_path}")
    print(f"Selected CSV: {selected.filename}")
    print(f"Extracted CSV: {extracted_path}")

    return run_author_mapping(
        input_path=extracted_path,
        output_dir=output_dir,
        authors_col=authors_col,
        full_name_col=full_name_col,
        title_col=title_col,
        year_col=year_col,
        chunksize=chunksize,
        max_rows=max_rows,
        progress_every=progress_every,
        create_zip=create_zip,
    )


## 7. Connect Google Drive

Run this cell when the ZIP file is stored in Google Drive.


In [ ]:
# Run in Colab when the file is stored in Google Drive:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
ZIP_PATH = "/content/df_with_openalex_fields_authors_tasks_fully_fixed.zip"
OUTPUT_DIR = "/content/drive/MyDrive/author_initials_mapping_results"

final_summary = run_mapping_from_zip(
    zip_path=ZIP_PATH,
    output_dir=OUTPUT_DIR,
    max_rows=None,
    chunksize=2000,
    progress_every=5000,
    create_zip=True,
)

from google.colab import files
files.download(final_summary["output_files"]["zip"])

In [ ]:
# After the complete run:

# full_name_mapping_df = pd.read_csv(
#     f"{OUTPUT_DIR}/full_name_initials_mapping.csv",
#     encoding="utf-8-sig",
# )
# unique_initials_df = pd.read_csv(
#     f"{OUTPUT_DIR}/unique_initials_mapping.csv",
#     encoding="utf-8-sig",
# )
# failures_df = pd.read_csv(
#     f"{OUTPUT_DIR}/initials_mapping_failures.csv",
#     encoding="utf-8-sig",
# )
# article_summary_df = pd.read_csv(
#     f"{OUTPUT_DIR}/article_mapping_summary.csv",
#     encoding="utf-8-sig",
# )
#
# display(full_name_mapping_df.head(20))
# display(unique_initials_df.head(20))
# display(failures_df.head(20))



# Part 2 — Nature: Full-Text Author Detection and Author–Task Assignment

This section reuses the **same alias-generation and collision-resolution rules defined for OnePLOS**. It does not create a second, simplified initials system.

The difference is the input structure:

- In OnePLOS, the `authors` field already contains the observed initials or abbreviated author forms.
- In Nature, the notebook searches the complete `contribution` paragraph for aliases of the article's full author names.
- After the author mentions are resolved, the parser detects whether each local structure is **authors → task** or **task → authors** and assigns the extracted task to every author in the corresponding author block.

The Nature stage therefore consists of:

1. Flattening the article-level full-author list.
2. Generating all aliases with the shared OnePLOS `build_alias_map` function.
3. Detecting the aliases inside the contribution paragraph while preserving their character spans.
4. Resolving collisions using the same specificity and longer-observed-alias logic.
5. Grouping adjacent author mentions into author blocks.
6. Extracting task text before or after each block.
7. Writing one raw author–task row per assignment and one aggregated row per article–author.


In [ ]:

# ============================================================
# PART 2A — NATURE AUTHOR DETECTION
# Reuses the OnePLOS alias-generation rules defined above.
# ============================================================

NATURE_AUTHOR_CONNECTOR_WORDS = {
    "and", "or", "both", "respectively", "et", "al",
    "with", "together", "as", "well", "also"
}

NATURE_ATTACH_CUES = {
    "by", "with", "from", "for", "using", "via", "through",
    "including", "under", "alongside"
}

NATURE_PURE_INITIAL_RULE_PREFIXES = (
    "initials_",
    "first_last_initials_",
    "three_token_special_",
    "particle_initials_",
    "compound_given_",
    "compound_given_reverse_",
)


def nature_coerce_list(value):
    """Convert a JSON/CSV value into a Python list without losing plain strings."""
    if isinstance(value, list):
        return value

    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []

    text = str(value).strip()
    if not text:
        return []

    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass

    return [value]


def nature_flatten_full_authors(author_value):
    """
    Flatten Nature author groups into one ordered article-level full-name list.

    The function supports:
    - a list of individual names;
    - a list containing comma/ampersand-separated author groups;
    - a string representation of a Python list.
    """
    items = nature_coerce_list(author_value)
    flattened = []
    seen = set()

    for item in items:
        if isinstance(item, list):
            parts = item
        else:
            text = norm_spaces(item)
            text = re.sub(r"\s+(?:&|and)\s+", ", ", text)
            parts = re.split(r"\s*,\s*", text)

        for part in parts:
            name = clean_raw_name(part)
            key = normalize_full_name_key(name)

            if key and key not in seen:
                seen.add(key)
                flattened.append(name)

    return flattened


def nature_alias_to_regex(alias_norm):
    """
    Build a flexible full-text pattern for one normalized alias.

    The same normalized alias can therefore match forms such as:
    AB, A.B., A B, JSmith, J. Smith, and John Smith.
    """
    chars = [re.escape(ch) for ch in alias_norm]
    between = r"(?:[.\s'’`–—-]*)"
    body = between.join(chars)

    return (
        r"(?<![A-Za-z])"
        + body
        + r"\.?"
        + r"(?![A-Za-z])"
    )


def nature_rule_requires_uppercase(rule):
    """Return True for aliases that represent initials rather than name words."""
    return str(rule).startswith(NATURE_PURE_INITIAL_RULE_PREFIXES)


def nature_rule_priority(rule):
    """
    Prefer an author's original name order when two authors share the same alias.

    Alternative orders and permutations remain valid fallback aliases, but they
    must not outrank an original-order match belonging to another author.
    """
    rule = str(rule)

    if "original_order" in rule or rule == "initials_original":
        return 3

    if any(
        marker in rule
        for marker in (
            "move_first_to_end",
            "move_last_to_front",
            "reverse_order",
            "subset_permutation",
            "initials_permutation",
        )
    ):
        return 1

    return 2


def nature_candidate_score(candidate):
    """Return the collision-resolution score for one Nature candidate."""
    return (
        candidate.get("rule_priority", nature_rule_priority(candidate.get("rule", ""))),
        candidate.get("specificity", 0),
    )


def nature_alias_surface_is_valid(alias_text, alias_norm, rule):
    """
    Reject ordinary lowercase words that happen to equal an initials alias.

    Pure-initial rules must appear in uppercase. Mixed surname/full-name aliases
    may contain lowercase letters, for example J. Smith or Chi-Fai Ng.
    """
    letters = "".join(re.findall(r"[A-Za-z]", str(alias_text)))

    if not letters:
        return False

    if compact(alias_text) != alias_norm:
        return False

    if alias_norm in COMMON_WORD_ALIAS_REQUIRES_UPPERCASE:
        return letters.isupper()

    if nature_rule_requires_uppercase(rule):
        return letters.isupper()

    return True


def build_nature_alias_rows(full_authors):
    """
    Generate searchable alias rows using the complete OnePLOS rule set.

    Returns one row per alias–author candidate and keeps the shared rule name
    and specificity score for collision resolution.
    """
    entities = build_name_entities(full_authors)
    entity_names = [entity["full_name"] for entity in entities]
    alias_map = build_alias_map(entity_names)

    rows = []

    for alias_norm, candidate_map in alias_map.items():
        pattern = nature_alias_to_regex(alias_norm)
        compiled = re.compile(pattern, flags=re.I)

        for entity_id, (specificity, rule) in candidate_map.items():
            entity = entities[entity_id]
            rows.append({
                "entity_id": entity_id,
                "author_index": entity["full_name_indices"][0],
                "full_name": entity["full_name"],
                "full_name_indices": list(entity["full_name_indices"]),
                "alias_norm": alias_norm,
                "specificity": specificity,
                "rule_priority": nature_rule_priority(rule),
                "rule": rule,
                "pattern": pattern,
                "compiled": compiled,
            })

    return rows, entities, alias_map


def collect_nature_alias_matches(text, alias_rows):
    """Find all valid author-alias occurrences and preserve their text spans."""
    text = norm_spaces(text)
    matches = []

    for alias in alias_rows:
        for match in alias["compiled"].finditer(text):
            alias_text = match.group(0).strip()

            if not nature_alias_surface_is_valid(
                alias_text,
                alias["alias_norm"],
                alias["rule"],
            ):
                continue

            matches.append({
                "span": match.span(),
                "alias_text": alias_text,
                "alias_norm": alias["alias_norm"],
                "entity_id": alias["entity_id"],
                "author_index": alias["author_index"],
                "full_name": alias["full_name"],
                "full_name_indices": list(alias["full_name_indices"]),
                "specificity": alias["specificity"],
                "rule_priority": alias["rule_priority"],
                "rule": alias["rule"],
            })

    return matches



## 2.1 Resolving Alias Collisions and Overlapping Matches

A short alias can correspond to several authors in the same article. The following code applies the same principles used in the OnePLOS stage:

- keep the most specific rule for each candidate author;
- use longer uniquely observed aliases to eliminate implausible candidates;
- prefer the author's original name order when it conflicts with an alternative-order alias from another author;
- then prefer the highest-specificity remaining candidate;
- preserve unresolved candidates and mark them for review;
- when text spans overlap, prefer the longest and most specific observed form.


In [ ]:
def nature_compute_unique_specific_aliases(all_matches):
    """Collect aliases that uniquely identify one author in the current article."""
    alias_to_entities = defaultdict(set)
    alias_to_matches = defaultdict(list)

    for match in all_matches:
        alias_to_entities[match["alias_norm"]].add(match["entity_id"])
        alias_to_matches[match["alias_norm"]].append(match)

    unique_specific = defaultdict(list)

    for alias_norm, entity_ids in alias_to_entities.items():
        if len(entity_ids) != 1:
            continue

        entity_id = next(iter(entity_ids))
        max_score = max(
            nature_candidate_score(item)
            for item in alias_to_matches[alias_norm]
        )
        unique_specific[entity_id].append(
            (alias_norm, max_score)
        )

    return unique_specific


def nature_resolve_candidate_group(candidates, unique_specific):
    """Resolve all author candidates attached to one observed alias span."""
    if not candidates:
        return [], "no_candidates"

    best_by_entity = {}

    for candidate in candidates:
        previous = best_by_entity.get(candidate["entity_id"])

        if (
            previous is None
            or nature_candidate_score(candidate)
            > nature_candidate_score(previous)
        ):
            best_by_entity[candidate["entity_id"]] = candidate

    candidates = list(best_by_entity.values())

    if len(candidates) == 1:
        return candidates, "unique"

    current_length = len(candidates[0]["alias_norm"])
    filtered = []

    for candidate in candidates:
        has_longer_unique_alias = any(
            len(observed_alias) > current_length
            and observed_score > nature_candidate_score(candidate)
            for observed_alias, observed_score
            in unique_specific.get(candidate["entity_id"], [])
        )

        if not has_longer_unique_alias:
            filtered.append(candidate)

    if len(filtered) == 1:
        return filtered, "resolved_by_longer_observed_alias"

    remaining = filtered if filtered else candidates
    max_score = max(
        nature_candidate_score(item)
        for item in remaining
    )
    top = [
        item for item in remaining
        if nature_candidate_score(item) == max_score
    ]

    if len(top) == 1:
        return top, "resolved_by_specificity"

    return top, "unresolved"


def nature_filter_overlapping_matches(matches):
    """
    Keep longer and more specific matches when textual spans overlap.

    Exact identical spans are retained when they represent unresolved candidates.
    """
    ordered = sorted(
        matches,
        key=lambda item: (
            -(item["span"][1] - item["span"][0]),
            -item.get("rule_priority", 0),
            -item.get("specificity", 0),
            item["span"][0],
        ),
    )

    accepted = []
    accepted_spans = []

    for item in ordered:
        start, end = item["span"]
        overlaps = any(
            not (end <= other_start or start >= other_end)
            for other_start, other_end in accepted_spans
        )
        exact_same = any(
            (start, end) == span
            for span in accepted_spans
        )

        if not overlaps or exact_same:
            accepted.append(item)
            accepted_spans.append((start, end))

    return sorted(
        accepted,
        key=lambda item: (
            item["span"][0],
            item["span"][1],
            item["entity_id"],
        ),
    )


def resolve_nature_alias_matches(
    text,
    all_matches,
    article_metadata,
):
    """
    Resolve span-level candidates and return both accepted matches and conflicts.
    """
    unique_specific = nature_compute_unique_specific_aliases(all_matches)
    grouped = defaultdict(list)

    for match in all_matches:
        grouped[(match["span"], match["alias_norm"])].append(match)

    resolved_matches = []
    conflicts = []

    for (span, alias_norm), candidates in grouped.items():
        resolved, status = nature_resolve_candidate_group(
            candidates,
            unique_specific,
        )

        if status == "unresolved":
            conflicts.append({
                **article_metadata,
                "alias_text": candidates[0]["alias_text"],
                "normalized_alias": alias_norm,
                "candidate_full_names": pipe_join(
                    sorted({item["full_name"] for item in candidates})
                ),
                "candidate_author_indices": pipe_join(
                    sorted({item["author_index"] for item in candidates})
                ),
                "status": status,
                "context": text[
                    max(0, span[0] - 100):
                    min(len(text), span[1] + 100)
                ],
            })

        for item in resolved:
            accepted = dict(item)
            accepted["resolution_status"] = status
            resolved_matches.append(accepted)

    return (
        nature_filter_overlapping_matches(resolved_matches),
        conflicts,
    )



## 2.2 Detecting Author Blocks and Extracting Tasks

The parser works sentence by sentence and does not assume one global direction for the entire paragraph.

Two adjacent author aliases belong to the same author block only when the text between them contains connectors or punctuation, rather than substantive task text. The parser then inspects the text on both sides of each block:

- `A.B. and C.D. designed the study` → **authors before task**
- `Data collection was performed by A.B. and C.D.` → **task before authors**
- `Conceptualization: A.B. and C.D.` → **task before authors**
- mixed clauses are split using the already detected author positions.

Commas do **not** split a task into several tasks at this stage. Multi-role CRediT classification remains a separate downstream stage.


In [ ]:

# ============================================================
# PART 2C — AUTHOR BLOCKS AND TASK EXTRACTION
# ============================================================

def nature_is_inside_span(position, spans):
    return any(start <= position < end for start, end in spans)


def nature_next_nonspace(text, position):
    while position < len(text) and text[position].isspace():
        position += 1
    return position if position < len(text) else None


def nature_alias_final_dot_is_boundary(text, match, all_matches):
    """
    Treat the final period of an alias as a sentence boundary only when it is
    not immediately followed by another detected author alias.
    """
    start, end = match["span"]

    if not match["alias_text"].endswith("."):
        return False

    next_position = nature_next_nonspace(text, end)

    if next_position is None:
        return True

    if any(
        other["span"][0] == next_position
        for other in all_matches
        if other is not match
    ):
        return False

    return text[next_position].isupper()


def build_nature_sentence_boundaries(text, matches):
    """Create sentence-like boundaries without splitting periods inside aliases."""
    alias_spans = [item["span"] for item in matches]
    boundaries = set()

    for index, char in enumerate(text):
        if char == ";":
            boundaries.add(index)
        elif char == "." and not nature_is_inside_span(index, alias_spans):
            boundaries.add(index)

    for match in matches:
        _, end = match["span"]

        if (
            end > 0
            and text[end - 1] == "."
            and nature_alias_final_dot_is_boundary(text, match, matches)
        ):
            boundaries.add(end - 1)

    return sorted(boundaries)


def nature_sentence_segments(text, boundaries):
    """Convert boundary positions into character-span sentence segments."""
    segments = []
    start = 0

    for boundary in boundaries:
        end = boundary + 1
        if end > start:
            segments.append((start, end))
        start = end

    if start < len(text):
        segments.append((start, len(text)))

    return segments


def nature_connector_only(text):
    """Return True when text contains only author-group connectors."""
    text = strip_accents(norm_spaces(text))

    if not text:
        return True

    cleaned = re.sub(
        r"[,.\s&/;:()\[\]{}\-–—]+",
        " ",
        text,
    ).strip().lower()

    if not cleaned:
        return True

    words = re.findall(r"[a-z]+", cleaned)
    return bool(words) and all(
        word in NATURE_AUTHOR_CONNECTOR_WORDS
        for word in words
    )


def nature_make_author_block(matches):
    """Create one ordered author block and remove duplicate candidates per author."""
    best_by_entity = {}

    for match in matches:
        previous = best_by_entity.get(match["entity_id"])

        if (
            previous is None
            or match.get("specificity", 0) > previous.get("specificity", 0)
        ):
            best_by_entity[match["entity_id"]] = match

    unique_matches = sorted(
        best_by_entity.values(),
        key=lambda item: item["span"][0],
    )

    return {
        "start": unique_matches[0]["span"][0],
        "end": unique_matches[-1]["span"][1],
        "matches": unique_matches,
    }


def build_nature_author_blocks(text, matches):
    """
    Group adjacent detected authors when only punctuation/connectors separate them.
    A repeated occurrence of the same author starts a new block.
    """
    matches = sorted(
        matches,
        key=lambda item: (
            item["span"][0],
            item["span"][1],
            item["entity_id"],
        ),
    )

    if not matches:
        return []

    blocks = []
    current = [matches[0]]
    seen_entities = {matches[0]["entity_id"]}

    for match in matches[1:]:
        gap = text[current[-1]["span"][1]:match["span"][0]]
        repeated_author = match["entity_id"] in seen_entities

        if nature_connector_only(gap) and not repeated_author:
            current.append(match)
            seen_entities.add(match["entity_id"])
        else:
            blocks.append(nature_make_author_block(current))
            current = [match]
            seen_entities = {match["entity_id"]}

    blocks.append(nature_make_author_block(current))
    return blocks


def nature_clean_task(task):
    """Normalize an extracted task span without splitting it on commas."""
    task = norm_spaces(task)
    task = task.replace("ג€”", "—")
    task = re.sub(r"\s*([—–-])\s*", r"\1", task)

    task = re.sub(
        r"(?i)^(author contributions?|contributions?)\s*:?\s*",
        "",
        task,
    )
    task = re.sub(r"^\s*:\s*", "", task)

    task = re.sub(
        r"(?i)^(and|or|both|by|were|was|respectively|and both)\b",
        "",
        task,
    ).strip()

    task = re.sub(
        r"(?i)\b(and|or|both|respectively)$",
        "",
        task,
    ).strip()

    for _ in range(4):
        cleaned = re.sub(
            r"(?i)\b(by|with|from|for|of|to|and|were|was|"
            r"together with|as well as)$",
            "",
            task,
        ).strip(" ,;:&")

        if cleaned == task:
            break
        task = cleaned

    return norm_spaces(task.strip(" .;,:&"))


def nature_task_is_bad(task):
    """Reject empty, initials-only, and non-informative task fragments."""
    task = nature_clean_task(task)

    if not task:
        return True

    letters = re.sub(r"[^A-Za-z]", "", strip_accents(task))

    if len(letters) < 3:
        return True

    if re.fullmatch(r"[A-Z]\.?", task):
        return True

    if re.fullmatch(
        r"(?:[A-Z](?:\.[A-Z]){0,7}\.?\s*(?:,|and|&)?\s*){1,50}",
        task,
    ):
        return True

    exact_bad = {
        "all authors",
        "authors",
        "author",
        "contribution",
        "contributions",
        "author contributions",
    }

    if task.lower() in exact_bad:
        return True

    words = re.findall(r"[A-Za-z]+", strip_accents(task))

    if words and all(
        word.lower() in NATURE_AUTHOR_CONNECTOR_WORDS
        for word in words
    ):
        return True

    return False


def nature_is_good_task(task):
    task = nature_clean_task(task)
    return bool(task) and not nature_task_is_bad(task)


def nature_task_ends_with_attach_cue(raw_task):
    raw_task = norm_spaces(raw_task).strip(" .;,:&")
    words = re.findall(
        r"[A-Za-z]+",
        strip_accents(raw_task.lower()),
    )
    return bool(words) and words[-1] in NATURE_ATTACH_CUES


def nature_starts_with_and(raw_text):
    return re.match(
        r"(?i)^\s*[,;:]?\s*and\b",
        raw_text or "",
    ) is not None


def nature_gap_before_block(text, blocks, block_index, sentence_start):
    start = (
        sentence_start
        if block_index == 0
        else blocks[block_index - 1]["end"]
    )
    return text[start:blocks[block_index]["start"]]


def nature_gap_after_block(text, blocks, block_index, sentence_end):
    end = (
        sentence_end
        if block_index + 1 == len(blocks)
        else blocks[block_index + 1]["start"]
    )
    return text[blocks[block_index]["end"]:end]


def nature_choose_mixed_split_index(text, block):
    """
    Split a combined block in a mixed structure such as:
    task -> A, B and C, D -> task
    """
    matches = block["matches"]
    count = len(matches)

    if count < 2:
        return None

    separators = [
        text[matches[index - 1]["span"][1]:matches[index]["span"][0]]
        for index in range(1, count)
    ]

    for index, separator in enumerate(separators, start=1):
        if (
            re.search(r"(?i)\band\b", separator)
            and index >= 2
            and (count - index) >= 3
        ):
            return index

    seen_and = False

    for index, separator in enumerate(separators, start=1):
        if re.search(r"(?i)\band\b", separator):
            seen_and = True
            continue

        if seen_and and re.fullmatch(r"\s*,\s*", separator or ""):
            return index

    return count - 1


def nature_author_block_text(text, block):
    return norm_spaces(
        text[block["start"]:block["end"]].strip(" ,;:")
    )


def nature_add_assignment_records(
    records,
    block,
    task,
    direction,
    text,
    metadata,
):
    """
    Assign one complete task span to every author in the detected author block.
    Commas inside the task are preserved.
    """
    task = nature_clean_task(task)

    if not nature_is_good_task(task):
        return

    if direction.startswith("authors_before"):
        raw_clause = norm_spaces(
            f"{nature_author_block_text(text, block)}, {task}"
        )
    else:
        raw_clause = norm_spaces(
            f"{task}, {nature_author_block_text(text, block)}"
        )

    contains_respectively = bool(
        re.search(r"(?i)\brespectively\b", raw_clause)
    )

    for match in block["matches"]:
        unresolved = (
            "unresolved"
            in str(match["resolution_status"]).lower()
        )

        records.append({
            **metadata,
            "full_name": match["full_name"],
            "author_index": match["author_index"],
            "full_name_indices": pipe_join(match["full_name_indices"]),
            "matched_alias": match["alias_text"],
            "normalized_alias": match["alias_norm"],
            "alias_rule": match["rule"],
            "alias_specificity": match["specificity"],
            "match_start": match["span"][0],
            "match_end": match["span"][1],
            "task": task,
            "raw_clause": raw_clause,
            "resolution_status": match["resolution_status"],
            "task_direction": direction,
            "contains_respectively": contains_respectively,
            "needs_review": unresolved or contains_respectively,
        })


def parse_nature_author_task_blocks(
    text,
    resolved_matches,
    boundaries,
    metadata,
):
    """
    Parse each sentence independently and support:
    - authors -> task;
    - task -> authors;
    - role label: authors;
    - mixed task -> authors, authors -> task structures.
    """
    records = []

    for sentence_start, sentence_end in nature_sentence_segments(
        text,
        boundaries,
    ):
        sentence_matches = [
            match for match in resolved_matches
            if (
                match["span"][0] >= sentence_start
                and match["span"][1] <= sentence_end
            )
        ]

        if not sentence_matches:
            continue

        blocks = build_nature_author_blocks(text, sentence_matches)

        for block_index, block in enumerate(blocks):
            left_raw = nature_gap_before_block(
                text,
                blocks,
                block_index,
                sentence_start,
            )
            right_raw = nature_gap_after_block(
                text,
                blocks,
                block_index,
                sentence_end,
            )

            left_task = nature_clean_task(left_raw)
            right_task = nature_clean_task(right_raw)

            left_good = nature_is_good_task(left_task)
            right_good = nature_is_good_task(right_task)

            left_cue = nature_task_ends_with_attach_cue(left_raw)
            right_cue = nature_task_ends_with_attach_cue(right_raw)

            right_is_bridge = (
                block_index + 1 < len(blocks)
                and nature_starts_with_and(right_raw)
                and right_cue
            )

            # Mixed structure:
            # task -> author_group_1, author_group_2 -> task
            if (
                left_good
                and right_good
                and left_cue
                and len(block["matches"]) >= 2
                and not right_is_bridge
            ):
                split_index = nature_choose_mixed_split_index(
                    text,
                    block,
                )

                if (
                    split_index is not None
                    and 0 < split_index < len(block["matches"])
                ):
                    left_block = nature_make_author_block(
                        block["matches"][:split_index]
                    )
                    right_block = nature_make_author_block(
                        block["matches"][split_index:]
                    )

                    nature_add_assignment_records(
                        records,
                        left_block,
                        left_task,
                        "task_before_authors_split",
                        text,
                        metadata,
                    )
                    nature_add_assignment_records(
                        records,
                        right_block,
                        right_task,
                        "authors_before_task_split",
                        text,
                        metadata,
                    )
                    continue

            # task -> authors, usually introduced by by/with/from/using
            if left_good and left_cue:
                nature_add_assignment_records(
                    records,
                    block,
                    left_task,
                    "task_before_authors",
                    text,
                    metadata,
                )

            # authors -> task
            if right_good and not right_is_bridge:
                nature_add_assignment_records(
                    records,
                    block,
                    right_task,
                    "authors_before_task",
                    text,
                    metadata,
                )

            # role label or task -> authors without an attachment cue
            if left_good and not left_cue and not right_good:
                nature_add_assignment_records(
                    records,
                    block,
                    left_task,
                    "task_before_authors",
                    text,
                    metadata,
                )

    return records


def extract_nature_all_authors_tasks(
    text,
    full_authors,
    metadata,
):
    """Assign a substantive 'All authors ...' clause to every article author."""
    records = []

    for match in re.finditer(r"\bAll authors\b", text, flags=re.I):
        sentence_end_candidates = [
            position for position in (
                text.find(".", match.end()),
                text.find(";", match.end()),
            )
            if position != -1
        ]
        sentence_end = (
            min(sentence_end_candidates)
            if sentence_end_candidates
            else len(text)
        )

        task = nature_clean_task(
            text[match.end():sentence_end]
        )

        if not nature_is_good_task(task):
            continue

        for author_index, full_name in enumerate(full_authors):
            records.append({
                **metadata,
                "full_name": full_name,
                "author_index": author_index,
                "full_name_indices": str(author_index),
                "matched_alias": "All authors",
                "normalized_alias": "allauthors",
                "alias_rule": "all_authors",
                "alias_specificity": 1000,
                "match_start": match.start(),
                "match_end": match.end(),
                "task": task,
                "raw_clause": norm_spaces(
                    text[match.start():sentence_end]
                ),
                "resolution_status": "all_authors",
                "task_direction": "all_authors",
                "contains_respectively": False,
                "needs_review": False,
            })

    return records


def nature_previous_boundary(boundaries, position):
    previous = [value for value in boundaries if value < position]
    return previous[-1] if previous else None


def nature_next_boundary(boundaries, position):
    following = [value for value in boundaries if value >= position]
    return following[0] if following else None


def nature_remove_aliases_from_segment(
    text,
    segment_start,
    segment_end,
    matches,
):
    """Remove detected author aliases before treating a span as task text."""
    segment = text[segment_start:segment_end]
    local_spans = []

    for match in matches:
        start, end = match["span"]

        if start >= segment_start and end <= segment_end:
            local_spans.append(
                (start - segment_start, end - segment_start)
            )

    for start, end in sorted(local_spans, reverse=True):
        segment = segment[:start] + " " + segment[end:]

    return segment


def nature_fallback_for_uncovered_matches(
    text,
    resolved_matches,
    boundaries,
    covered_spans,
    metadata,
):
    """Use local before/after context only for mentions not handled by blocks."""
    records = []

    for match in resolved_matches:
        if match["span"] in covered_spans:
            continue

        start, end = match["span"]

        right_boundary = nature_next_boundary(boundaries, end)
        right_end = (
            right_boundary
            if right_boundary is not None
            else len(text)
        )
        right_task = nature_clean_task(
            nature_remove_aliases_from_segment(
                text,
                end,
                right_end,
                resolved_matches,
            )
        )

        left_boundary = nature_previous_boundary(boundaries, start)
        left_start = (
            left_boundary + 1
            if left_boundary is not None
            else 0
        )
        left_task = nature_clean_task(
            nature_remove_aliases_from_segment(
                text,
                left_start,
                start,
                resolved_matches,
            )
        )

        if nature_is_good_task(right_task):
            block = nature_make_author_block([match])
            nature_add_assignment_records(
                records,
                block,
                right_task,
                "authors_before_task_fallback",
                text,
                metadata,
            )
        elif nature_is_good_task(left_task):
            block = nature_make_author_block([match])
            nature_add_assignment_records(
                records,
                block,
                left_task,
                "task_before_authors_fallback",
                text,
                metadata,
            )

    return records


def extract_nature_article(
    contribution_text,
    author_value,
    metadata,
):
    """
    Run the complete Nature author detection and task-assignment pipeline
    for one article.
    """
    text = norm_spaces(contribution_text)
    full_authors = nature_flatten_full_authors(author_value)

    article_summary = {
        **metadata,
        "total_authors": len(full_authors),
        "detected_alias_occurrences": 0,
        "resolved_alias_occurrences": 0,
        "matched_authors": 0,
        "unmatched_authors": len(full_authors),
        "raw_author_task_assignments": 0,
        "unique_extracted_tasks": 0,
        "alias_conflicts": 0,
        "article_needs_review": False,
        "processing_status": "",
    }

    if not text:
        article_summary["processing_status"] = "empty_contribution"
        return [], [], article_summary

    if not full_authors:
        article_summary["processing_status"] = "empty_author_list"
        return [], [], article_summary

    alias_rows, entities, _ = build_nature_alias_rows(full_authors)
    all_matches = collect_nature_alias_matches(text, alias_rows)

    resolved_matches, conflicts = resolve_nature_alias_matches(
        text,
        all_matches,
        metadata,
    )

    boundaries = build_nature_sentence_boundaries(
        text,
        resolved_matches,
    )

    records = extract_nature_all_authors_tasks(
        text,
        full_authors,
        metadata,
    )

    block_records = parse_nature_author_task_blocks(
        text,
        resolved_matches,
        boundaries,
        metadata,
    )
    records.extend(block_records)

    covered_spans = {
        (row["match_start"], row["match_end"])
        for row in block_records
    }

    records.extend(
        nature_fallback_for_uncovered_matches(
            text,
            resolved_matches,
            boundaries,
            covered_spans,
            metadata,
        )
    )

    deduplicated = []
    seen = set()

    for row in records:
        key = (
            row["article_instance_key"],
            row["full_name"],
            row["matched_alias"],
            row["task"],
            row["raw_clause"],
        )

        if key not in seen:
            seen.add(key)
            deduplicated.append(row)

    matched_authors = {
        row["full_name"]
        for row in deduplicated
    }
    unique_tasks = {
        nature_clean_task(row["task"])
        for row in deduplicated
        if nature_clean_task(row["task"])
    }

    article_summary.update({
        "detected_alias_occurrences": len({
            (item["span"], item["alias_norm"])
            for item in all_matches
        }),
        "resolved_alias_occurrences": len({
            (item["span"], item["alias_norm"], item["entity_id"])
            for item in resolved_matches
        }),
        "matched_authors": len(matched_authors),
        "unmatched_authors": max(
            0,
            len(full_authors) - len(matched_authors),
        ),
        "raw_author_task_assignments": len(deduplicated),
        "unique_extracted_tasks": len(unique_tasks),
        "alias_conflicts": len(conflicts),
        "article_needs_review": (
            bool(conflicts)
            or any(row["needs_review"] for row in deduplicated)
        ),
        "processing_status": (
            "processed_with_assignments"
            if deduplicated
            else "processed_no_assignments"
        ),
    })

    return deduplicated, conflicts, article_summary



## 2.3 Inspect One Nature Article

Use this helper before the full run. It displays:

1. raw author–task assignments;
2. unresolved alias conflicts;
3. an article-level extraction summary.

The example deliberately contains both directions in one paragraph.


In [ ]:

def inspect_one_nature_article(
    contribution_text,
    full_authors,
    title="Example Nature article",
    year="",
    url="",
):
    """Run and display the complete Nature parser for one article."""
    metadata = {
        "dataset": "Nature",
        "source_folder": "example",
        "file_name": "example.json",
        "file_path": "",
        "source_key": "example",
        "local_article_index": 0,
        "global_article_index": 0,
        "article_instance_key": "example||0",
        "title": title,
        "year": year,
        "url": url,
    }

    records, conflicts, summary = extract_nature_article(
        contribution_text,
        full_authors,
        metadata,
    )

    raw_table = pd.DataFrame(records)
    conflict_table = pd.DataFrame(conflicts)
    summary_table = pd.DataFrame([summary])

    print("Raw author-task assignments")
    display(raw_table)

    print("Alias conflicts")
    display(conflict_table)

    print("Article summary")
    display(summary_table)

    return raw_table, conflict_table, summary_table


nature_example_text = (
    "A.B. and C.D. designed the study. "
    "Data collection was performed by E.F. and G.H.; "
    "Writing—original draft: A.B. and E.F."
)

nature_example_authors = [
    "Alice Brown",
    "Charles Davis",
    "Emily Foster",
    "George Hall",
]

nature_example_raw, nature_example_conflicts, nature_example_summary = (
    inspect_one_nature_article(
        nature_example_text,
        nature_example_authors,
    )
)


Raw author-task assignments


,dataset,source_folder,file_name,file_path,source_key,local_article_index,global_article_index,article_instance_key,title,year,...,alias_rule,alias_specificity,match_start,match_end,task,raw_clause,resolution_status,task_direction,contains_respectively,needs_review
0,Nature,example,example.json,,example,0,0,example||0,Example Nature article,,...,initials_original_order,2,0,4,designed the study,"A.B. and C.D., designed the study",unique,authors_before_task,False,False
1,Nature,example,example.json,,example,0,0,example||0,Example Nature article,,...,initials_original_order,2,9,13,designed the study,"A.B. and C.D., designed the study",unique,authors_before_task,False,False
2,Nature,example,example.json,,example,0,0,example||0,Example Nature article,,...,initials_original_order,2,67,71,Data collection was performed,"Data collection was performed, E.F. and G.H.",unique,task_before_authors,False,False
3,Nature,example,example.json,,example,0,0,example||0,Example Nature article,,...,initials_original_order,2,76,80,Data collection was performed,"Data collection was performed, E.F. and G.H.",unique,task_before_authors,False,False
4,Nature,example,example.json,,example,0,0,example||0,Example Nature article,,...,initials_original_order,2,106,110,Writing—original draft,"Writing—original draft, A.B. and E.F.",unique,task_before_authors,False,False
5,Nature,example,example.json,,example,0,0,example||0,Example Nature article,,...,initials_original_order,2,115,119,Writing—original draft,"Writing—original draft, A.B. and E.F.",unique,task_before_authors,False,False


Alias conflicts


""


Article summary


,dataset,source_folder,file_name,file_path,source_key,local_article_index,global_article_index,article_instance_key,title,year,...,total_authors,detected_alias_occurrences,resolved_alias_occurrences,matched_authors,unmatched_authors,raw_author_task_assignments,unique_extracted_tasks,alias_conflicts,article_needs_review,processing_status
0,Nature,example,example.json,,example,0,0,example||0,Example Nature article,,...,4,6,6,4,0,6,3,0,False,processed_with_assignments



## 2.4 Process All Nature JSON Files

The runner below is designed for the Nature JSON structure used in this project:

```python
data[source_key]["contribution"][article_index]
data[source_key]["authors"][article_index]
data[source_key]["title"][article_index]
data[source_key]["year"][article_index]
data[source_key]["url"][article_index]
```

It writes results incrementally instead of holding the complete corpus in memory. Two output versions are produced:

- **all articles**;
- **unique articles by normalized title**, keeping the first occurrence.

The article summary also provides the total number of author instances, matched authors, extracted assignments, conflicts, and processing failures.


In [ ]:
NATURE_RAW_FIELDS = [
    "dataset", "source_folder", "file_name", "file_path",
    "source_key", "local_article_index", "global_article_index",
    "article_instance_key", "title", "year", "url",
    "full_name", "author_index", "full_name_indices",
    "matched_alias", "normalized_alias", "alias_rule",
    "alias_specificity", "match_start", "match_end",
    "task", "raw_clause", "resolution_status", "task_direction",
    "contains_respectively", "needs_review",
]

NATURE_CONFLICT_FIELDS = [
    "dataset", "source_folder", "file_name", "file_path",
    "source_key", "local_article_index", "global_article_index",
    "article_instance_key", "title", "year", "url",
    "alias_text", "normalized_alias", "candidate_full_names",
    "candidate_author_indices", "status", "context",
]

NATURE_ARTICLE_FIELDS = [
    "dataset", "source_folder", "file_name", "file_path",
    "source_key", "local_article_index", "global_article_index",
    "article_instance_key", "title", "year", "url",
    "total_authors", "detected_alias_occurrences",
    "resolved_alias_occurrences", "matched_authors",
    "unmatched_authors", "raw_author_task_assignments",
    "unique_extracted_tasks", "alias_conflicts",
    "article_needs_review", "processing_status",
]

NATURE_MAPPING_FIELDS = [
    "dataset", "source_folder", "file_name", "file_path",
    "source_key", "local_article_index", "global_article_index",
    "article_instance_key", "title", "year", "url",
    "full_name", "author_index", "full_name_indices",
    "tasks", "n_tasks", "matched_aliases", "alias_rules",
    "raw_clauses", "resolution_statuses", "task_directions",
    "contains_respectively", "needs_review",
]

NATURE_ERROR_FIELDS = [
    "source_folder", "file_name", "file_path",
    "source_key", "local_article_index",
    "global_article_index", "title", "error",
]


def nature_normalize_title(title):
    """Normalize article titles for cross-file duplicate removal."""
    title = strip_accents(norm_spaces(title)).lower()
    title = re.sub(r"[^\w\s]", "", title)
    return norm_spaces(title)


def nature_safe_sequence_value(container, key, index, default=""):
    """Read one indexed JSON value without raising on missing metadata fields."""
    sequence = container.get(key, [])

    if not isinstance(sequence, list) or index >= len(sequence):
        return default

    return sequence[index]


def nature_sorted_keys(keys):
    def sort_key(value):
        return (
            (0, int(value))
            if str(value).isdigit()
            else (1, str(value))
        )
    return sorted(keys, key=sort_key)


def nature_append_unique(values, value):
    value = norm_spaces(value)
    if value and value not in values:
        values.append(value)


def nature_mapping_rows_for_article(records):
    """Aggregate raw assignments into one row per article–author."""
    groups = {}

    for row in records:
        key = (
            row["article_instance_key"],
            row["full_name"],
            row["author_index"],
        )

        if key not in groups:
            groups[key] = {
                field: row.get(field, "")
                for field in [
                    "dataset", "source_folder", "file_name", "file_path",
                    "source_key", "local_article_index",
                    "global_article_index", "article_instance_key",
                    "title", "year", "url", "full_name",
                    "author_index", "full_name_indices",
                ]
            }
            groups[key].update({
                "tasks_list": [],
                "aliases_list": [],
                "rules_list": [],
                "clauses_list": [],
                "statuses_list": [],
                "directions_list": [],
                "contains_respectively": False,
                "needs_review": False,
            })

        group = groups[key]
        nature_append_unique(group["tasks_list"], row["task"])
        nature_append_unique(group["aliases_list"], row["matched_alias"])
        nature_append_unique(group["rules_list"], row["alias_rule"])
        nature_append_unique(group["clauses_list"], row["raw_clause"])
        nature_append_unique(
            group["statuses_list"],
            row["resolution_status"],
        )
        nature_append_unique(
            group["directions_list"],
            row["task_direction"],
        )
        group["contains_respectively"] = (
            group["contains_respectively"]
            or bool(row["contains_respectively"])
        )
        group["needs_review"] = (
            group["needs_review"]
            or bool(row["needs_review"])
        )

    output = []

    for group in groups.values():
        output.append({
            **{
                field: group[field]
                for field in [
                    "dataset", "source_folder", "file_name", "file_path",
                    "source_key", "local_article_index",
                    "global_article_index", "article_instance_key",
                    "title", "year", "url", "full_name",
                    "author_index", "full_name_indices",
                ]
            },
            "tasks": pipe_join(group["tasks_list"]),
            "n_tasks": len(group["tasks_list"]),
            "matched_aliases": pipe_join(group["aliases_list"]),
            "alias_rules": pipe_join(group["rules_list"]),
            "raw_clauses": pipe_join(group["clauses_list"]),
            "resolution_statuses": pipe_join(group["statuses_list"]),
            "task_directions": pipe_join(group["directions_list"]),
            "contains_respectively": group["contains_respectively"],
            "needs_review": group["needs_review"],
        })

    return sorted(
        output,
        key=lambda row: (
            row["global_article_index"],
            row["author_index"],
        ),
    )


def nature_open_writer(path, fields):
    """Open one UTF-8-SIG CSV and immediately write its header."""
    handle = open(path, "w", newline="", encoding="utf-8-sig")
    writer = csv.DictWriter(
        handle,
        fieldnames=fields,
        extrasaction="ignore",
    )
    writer.writeheader()
    return handle, writer


def run_nature_json_folders(
    input_folders,
    output_dir="/content/nature_author_task_results",
    limit_files=None,
    limit_articles_per_key=None,
    progress_every=500,
    create_zip=True,
):
    """
    Process every Nature JSON file with the shared OnePLOS alias rules.

    The function streams all outputs to disk and therefore does not retain the
    complete corpus in memory.
    """
    started = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    json_files = []

    for folder in input_folders:
        folder_path = Path(folder)

        if not folder_path.exists():
            print(f"WARNING: folder does not exist: {folder_path}")
            continue

        json_files.extend(sorted(folder_path.glob("*.json")))

    if limit_files is not None:
        json_files = json_files[:int(limit_files)]

    if not json_files:
        raise FileNotFoundError(
            "No JSON files were found in the supplied Nature folders."
        )

    paths = {
        "raw_all": output_dir / "nature_raw_author_task_assignments_all_articles.csv",
        "mapping_all": output_dir / "nature_author_task_mapping_all_articles.csv",
        "conflicts_all": output_dir / "nature_alias_conflicts_all_articles.csv",
        "articles_all": output_dir / "nature_article_summary_all_articles.csv",
        "raw_unique": output_dir / "nature_raw_author_task_assignments_unique_articles.csv",
        "mapping_unique": output_dir / "nature_author_task_mapping_unique_articles.csv",
        "conflicts_unique": output_dir / "nature_alias_conflicts_unique_articles.csv",
        "articles_unique": output_dir / "nature_article_summary_unique_articles.csv",
        "errors": output_dir / "nature_processing_errors.csv",
        "summary": output_dir / "nature_run_summary.json",
    }

    opened = {}
    opened["raw_all"] = nature_open_writer(paths["raw_all"], NATURE_RAW_FIELDS)
    opened["mapping_all"] = nature_open_writer(paths["mapping_all"], NATURE_MAPPING_FIELDS)
    opened["conflicts_all"] = nature_open_writer(paths["conflicts_all"], NATURE_CONFLICT_FIELDS)
    opened["articles_all"] = nature_open_writer(paths["articles_all"], NATURE_ARTICLE_FIELDS)
    opened["raw_unique"] = nature_open_writer(paths["raw_unique"], NATURE_RAW_FIELDS)
    opened["mapping_unique"] = nature_open_writer(paths["mapping_unique"], NATURE_MAPPING_FIELDS)
    opened["conflicts_unique"] = nature_open_writer(paths["conflicts_unique"], NATURE_CONFLICT_FIELDS)
    opened["articles_unique"] = nature_open_writer(paths["articles_unique"], NATURE_ARTICLE_FIELDS)
    opened["errors"] = nature_open_writer(paths["errors"], NATURE_ERROR_FIELDS)

    counters = Counter()
    status_counts = Counter()
    seen_title_keys = set()
    global_article_index = 0

    try:
        for file_number, json_path in enumerate(json_files, start=1):
            source_folder = json_path.parent.name

            print(
                f"\nFILE {file_number}/{len(json_files)}: "
                f"{json_path.name}"
            )

            with open(json_path, "r", encoding="utf-8") as handle:
                data = json.load(handle)

            for source_key in nature_sorted_keys(data.keys()):
                key_data = data[source_key]

                if (
                    not isinstance(key_data, dict)
                    or not isinstance(key_data.get("contribution"), list)
                ):
                    continue

                article_count = len(key_data["contribution"])

                if limit_articles_per_key is not None:
                    article_count = min(
                        article_count,
                        int(limit_articles_per_key),
                    )

                for local_article_index in range(article_count):
                    title = nature_safe_sequence_value(
                        key_data,
                        "title",
                        local_article_index,
                        "",
                    )
                    year = nature_safe_sequence_value(
                        key_data,
                        "year",
                        local_article_index,
                        "",
                    )
                    url = nature_safe_sequence_value(
                        key_data,
                        "url",
                        local_article_index,
                        "",
                    )
                    contribution = nature_safe_sequence_value(
                        key_data,
                        "contribution",
                        local_article_index,
                        "",
                    )
                    authors = nature_safe_sequence_value(
                        key_data,
                        "authors",
                        local_article_index,
                        [],
                    )

                    article_instance_key = (
                        f"{source_folder}||{json_path.name}||"
                        f"{source_key}||{local_article_index}"
                    )

                    metadata = {
                        "dataset": "Nature",
                        "source_folder": source_folder,
                        "file_name": json_path.name,
                        "file_path": str(json_path),
                        "source_key": source_key,
                        "local_article_index": local_article_index,
                        "global_article_index": global_article_index,
                        "article_instance_key": article_instance_key,
                        "title": title,
                        "year": year,
                        "url": url,
                    }

                    try:
                        records, conflicts, article_summary = (
                            extract_nature_article(
                                contribution,
                                authors,
                                metadata,
                            )
                        )

                        mapping_rows = nature_mapping_rows_for_article(
                            records
                        )

                        opened["raw_all"][1].writerows(records)
                        opened["mapping_all"][1].writerows(mapping_rows)
                        opened["conflicts_all"][1].writerows(conflicts)
                        opened["articles_all"][1].writerow(article_summary)

                        normalized_title = nature_normalize_title(title)
                        dedup_key = (
                            normalized_title
                            if normalized_title
                            else article_instance_key
                        )
                        is_unique_title = dedup_key not in seen_title_keys

                        if is_unique_title:
                            seen_title_keys.add(dedup_key)
                            opened["raw_unique"][1].writerows(records)
                            opened["mapping_unique"][1].writerows(mapping_rows)
                            opened["conflicts_unique"][1].writerows(conflicts)
                            opened["articles_unique"][1].writerow(article_summary)

                        counters["articles_total"] += 1
                        counters["unique_article_titles"] += int(is_unique_title)
                        counters["author_instances_total"] += article_summary["total_authors"]
                        counters["matched_author_instances"] += article_summary["matched_authors"]
                        counters["unmatched_author_instances"] += article_summary["unmatched_authors"]
                        counters["raw_author_task_assignments"] += len(records)
                        counters["article_author_mapping_rows"] += len(mapping_rows)
                        counters["alias_conflicts"] += len(conflicts)
                        counters["articles_needing_review"] += int(
                            article_summary["article_needs_review"]
                        )
                        counters["respectively_assignments"] += sum(
                            bool(row["contains_respectively"])
                            for row in records
                        )
                        status_counts[
                            article_summary["processing_status"]
                        ] += 1

                    except Exception as error:
                        counters["errors"] += 1
                        opened["errors"][1].writerow({
                            "source_folder": source_folder,
                            "file_name": json_path.name,
                            "file_path": str(json_path),
                            "source_key": source_key,
                            "local_article_index": local_article_index,
                            "global_article_index": global_article_index,
                            "title": title,
                            "error": repr(error),
                        })

                    global_article_index += 1

                    if (
                        progress_every
                        and global_article_index % progress_every == 0
                    ):
                        elapsed = time.time() - started
                        rate = (
                            global_article_index / elapsed
                            if elapsed > 0
                            else 0
                        )
                        print(
                            f"Articles: {global_article_index:,} | "
                            f"Assignments: "
                            f"{counters['raw_author_task_assignments']:,} | "
                            f"Conflicts: {counters['alias_conflicts']:,} | "
                            f"Errors: {counters['errors']:,} | "
                            f"Speed: {rate:.2f} articles/sec"
                        )

    finally:
        for handle, _ in opened.values():
            handle.close()

    elapsed = time.time() - started
    total_authors = counters["author_instances_total"]
    matched_authors = counters["matched_author_instances"]

    summary = {
        "input_json_files": [str(path) for path in json_files],
        "counts": dict(counters),
        "processing_status": dict(status_counts),
        "author_detection": {
            "total_author_instances": total_authors,
            "matched_author_instances": matched_authors,
            "unmatched_author_instances": counters[
                "unmatched_author_instances"
            ],
            "matched_author_rate_percent": (
                100 * matched_authors / total_authors
                if total_authors
                else 0.0
            ),
        },
        "elapsed_minutes": elapsed / 60,
        "output_files": {
            name: str(path)
            for name, path in paths.items()
        },
    }

    with open(paths["summary"], "w", encoding="utf-8") as handle:
        json.dump(
            summary,
            handle,
            ensure_ascii=False,
            indent=2,
        )

    zip_path = None

    if create_zip:
        zip_path = shutil.make_archive(
            str(output_dir),
            "zip",
            root_dir=output_dir,
        )
        summary["output_zip"] = zip_path

    print("\nNATURE PROCESSING COMPLETE")
    print("-" * 70)
    print(f"JSON files: {len(json_files):,}")
    print(f"Articles: {counters['articles_total']:,}")
    print(f"Unique article titles: {counters['unique_article_titles']:,}")
    print(f"Author instances: {total_authors:,}")
    print(
        "Raw author-task assignments: "
        f"{counters['raw_author_task_assignments']:,}"
    )
    print(f"Alias conflicts: {counters['alias_conflicts']:,}")
    print(f"Errors: {counters['errors']:,}")
    print(f"Output directory: {output_dir}")

    if zip_path:
        print(f"Output ZIP: {zip_path}")

    return summary



## 2.5 Run All

Set `RUN_NATURE = True` after confirming the folder paths. For a small validation run, set `LIMIT_FILES = 1` and `LIMIT_ARTICLES_PER_KEY = 10`.


In [ ]:

NATURE_INPUT_FOLDERS = [
    "/content/drive/MyDrive/data fro data_mining_project/nature/",
    "/content/drive/MyDrive/data fro data_mining_project/nature communication/",
]

NATURE_OUTPUT_DIR = (
    "/content/outputs_nature_shared_oneplos_rules/"
)

RUN_NATURE = True

LIMIT_FILES = None
LIMIT_ARTICLES_PER_KEY = None
NATURE_PROGRESS_EVERY = 500

if RUN_NATURE:
    nature_run_summary = run_nature_json_folders(
        input_folders=NATURE_INPUT_FOLDERS,
        output_dir=NATURE_OUTPUT_DIR,
        limit_files=LIMIT_FILES,
        limit_articles_per_key=LIMIT_ARTICLES_PER_KEY,
        progress_every=NATURE_PROGRESS_EVERY,
        create_zip=True,
    )



FILE 1/21:  humanities and social sciences communications_updated.json
Articles: 500 | Assignments: 3,108 | Conflicts: 49 | Errors: 0 | Speed: 9.09 articles/sec
Articles: 1,000 | Assignments: 5,790 | Conflicts: 84 | Errors: 0 | Speed: 10.82 articles/sec
Articles: 1,500 | Assignments: 7,330 | Conflicts: 116 | Errors: 0 | Speed: 12.64 articles/sec
Articles: 2,000 | Assignments: 8,192 | Conflicts: 125 | Errors: 0 | Speed: 14.56 articles/sec

FILE 2/21:  light: science and applications _updated.json
Articles: 2,500 | Assignments: 10,183 | Conflicts: 195 | Errors: 0 | Speed: 14.42 articles/sec
Articles: 3,000 | Assignments: 18,708 | Conflicts: 440 | Errors: 0 | Speed: 10.50 articles/sec
Articles: 3,500 | Assignments: 23,807 | Conflicts: 566 | Errors: 0 | Speed: 9.79 articles/sec

FILE 3/21:  prostate cancer and prostatic diseases_updated.json
Articles: 4,000 | Assignments: 28,547 | Conflicts: 665 | Errors: 0 | Speed: 8.63 articles/sec
Articles: 4,500 | Assignments: 28,944 | Conflicts: 668 

In [ ]:
import csv
import re
import unicodedata
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path

ORIGINAL_CSV = Path("/content/nature_author_task_mapping_all_articles.csv")
RESOLVED_CSV = Path("/content/nature_author_task_mapping_all_articles_advanced_fixed.csv")

OUTPUT_CSV = Path(
    "/content/nature_author_task_mapping_unique_articles_advanced_task_aligned.csv"
)


STOPWORDS = {
    "a", "an", "the", "and", "or", "of", "to", "for", "in", "on", "at",
    "by", "with", "from", "as", "was", "were", "is", "are", "be", "been",
    "being", "this", "that", "these", "those", "their", "his", "her",
}

TRANSLATION = str.maketrans({
    "ß": "ss", "ẞ": "SS", "ø": "o", "Ø": "O", "æ": "ae", "Æ": "AE",
    "ł": "l", "Ł": "L", "đ": "d", "Đ": "D", "ð": "d", "Ð": "D",
    "þ": "th", "Þ": "TH",
})


def split_pipe(value):
    return [
        item.strip()
        for item in str(value or "").split("|")
        if item.strip()
    ]


def join_pipe(values):
    return " | ".join(
        dict.fromkeys(
            value.strip()
            for value in values
            if value and value.strip()
        )
    )


def strip_accents(value):
    value = str(value).translate(TRANSLATION)
    return "".join(
        char
        for char in unicodedata.normalize("NFKD", value)
        if not unicodedata.combining(char)
    )


def normalize_text(value):
    return re.sub(
        r"[^a-z0-9]+",
        " ",
        strip_accents(value).lower()
    ).strip()


def content_tokens(value):
    return [
        token
        for token in normalize_text(value).split()
        if token not in STOPWORDS
    ]


def task_clause_score(task, clause):
    """
    Return a score in [0, 1] describing how strongly a task is supported
    by a source clause.
    """
    task_norm = normalize_text(task)
    clause_norm = normalize_text(clause)

    if not task_norm or not clause_norm:
        return 0.0

    # Strongest evidence: the task text occurs in the source clause.
    if re.search(
        rf"(?<![a-z0-9]){re.escape(task_norm)}(?![a-z0-9])",
        clause_norm,
    ):
        return 1.0

    task_tokens = content_tokens(task)
    clause_tokens = content_tokens(clause)

    if not task_tokens or not clause_tokens:
        return 0.0

    task_set = set(task_tokens)
    clause_set = set(clause_tokens)

    coverage = len(task_set & clause_set) / len(task_set)
    precision = len(task_set & clause_set) / len(clause_set)

    # Useful for slightly different punctuation or wording.
    sequence_score = SequenceMatcher(
        None,
        task_norm,
        clause_norm
    ).ratio()

    # One-word CRediT roles must appear as a complete token.
    if len(task_set) == 1:
        return 0.95 if next(iter(task_set)) in clause_set else 0.0

    return max(
        coverage,
        0.75 * coverage + 0.25 * precision,
        sequence_score,
    )


def row_key(row):
    return (
        row.get("article_instance_key", ""),
        row.get("full_name", ""),
        row.get("author_index", ""),
    )


def read_csv(path):
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        return list(reader), list(reader.fieldnames or [])


def align_tasks(original_row, resolved_row):
    """
    Align tasks with the clauses retained after alias conflict resolution.

    Unresolved aliases retain the same clauses and therefore the same tasks.
    A task is removed only when its supporting source clauses were reassigned
    to another author.
    """
    original_tasks = split_pipe(original_row.get("tasks", ""))
    original_clauses = split_pipe(original_row.get("raw_clauses", ""))
    retained_clauses = split_pipe(resolved_row.get("raw_clauses", ""))

    retained_norms = {
        normalize_text(clause)
        for clause in retained_clauses
    }

    removed_clauses = [
        clause
        for clause in original_clauses
        if normalize_text(clause) not in retained_norms
    ]

    aliases_changed = (
        original_row.get("matched_aliases", "")
        != resolved_row.get("matched_aliases", "")
    )

    kept_tasks = []
    removed_tasks = []
    uncertain_tasks = []

    for task in original_tasks:
        retained_scores = [
            task_clause_score(task, clause)
            for clause in retained_clauses
        ]
        removed_scores = [
            task_clause_score(task, clause)
            for clause in removed_clauses
        ]

        best_retained = max(retained_scores, default=0.0)
        best_removed = max(removed_scores, default=0.0)

        # Strong support from at least one clause still assigned to the author.
        if best_retained >= 0.72:
            kept_tasks.append(task)
            continue

        # The task is supported only by a clause transferred to another author.
        if (
            best_removed >= 0.72
            and best_removed >= best_retained + 0.12
        ):
            removed_tasks.append(task)
            continue

        # If all clauses were transferred, no task can remain on this row.
        if aliases_changed and not retained_clauses:
            removed_tasks.append(task)
            continue

        # No safe source reconstruction: preserve rather than invent/delete.
        kept_tasks.append(task)
        uncertain_tasks.append(task)

    status = "unchanged"

    if removed_tasks:
        status = "tasks_realigned"
    elif aliases_changed:
        status = "aliases_changed_tasks_verified"

    if uncertain_tasks:
        status += "_with_uncertain_tasks"

    return {
        "tasks": join_pipe(kept_tasks),
        "n_tasks": str(len(kept_tasks)),
        "removed_tasks": join_pipe(removed_tasks),
        "uncertain_tasks": join_pipe(uncertain_tasks),
        "status": status,
    }


def run_alignment():
    original_rows, _ = read_csv(ORIGINAL_CSV)
    resolved_rows, resolved_fields = read_csv(RESOLVED_CSV)

    original_map = {
        row_key(row): row
        for row in original_rows
    }

    output_fields = list(resolved_fields)

    for field in [
        "tasks_removed_after_alias_resolution",
        "task_alignment_uncertain",
        "task_alignment_status",
    ]:
        if field not in output_fields:
            output_fields.append(field)

    report_fields = [
        "article_instance_key",
        "title",
        "full_name",
        "author_index",
        "aliases_before",
        "aliases_after",
        "tasks_before",
        "tasks_after",
        "tasks_removed",
        "uncertain_tasks",
        "task_alignment_status",
        "postprocess_rules",
        "postprocess_confidence",
    ]

    output_rows = []
    report_rows = []
    summary = Counter()

    for resolved_row in resolved_rows:
        key = row_key(resolved_row)
        original_row = original_map.get(key)

        if original_row is None:
            # Preserve unmatched rows without modification.
            resolved_row["tasks_removed_after_alias_resolution"] = ""
            resolved_row["task_alignment_uncertain"] = ""
            resolved_row["task_alignment_status"] = "missing_original_row"
            output_rows.append(resolved_row)
            summary["missing_original_rows"] += 1
            continue

        aligned = align_tasks(original_row, resolved_row)

        tasks_before = original_row.get("tasks", "")
        tasks_after = aligned["tasks"]

        resolved_row["tasks"] = tasks_after
        resolved_row["n_tasks"] = aligned["n_tasks"]
        resolved_row[
            "tasks_removed_after_alias_resolution"
        ] = aligned["removed_tasks"]
        resolved_row[
            "task_alignment_uncertain"
        ] = aligned["uncertain_tasks"]
        resolved_row[
            "task_alignment_status"
        ] = aligned["status"]

        output_rows.append(resolved_row)

        summary["rows"] += 1

        if (
            original_row.get("matched_aliases", "")
            != resolved_row.get("matched_aliases", "")
        ):
            summary["rows_with_alias_changes"] += 1

        if tasks_before != tasks_after:
            summary["rows_with_task_changes"] += 1

        if aligned["removed_tasks"]:
            summary["rows_with_tasks_removed"] += 1
            summary["tasks_removed"] += len(
                split_pipe(aligned["removed_tasks"])
            )

        if aligned["uncertain_tasks"]:
            summary["rows_with_uncertain_tasks"] += 1

        if (
            original_row.get("matched_aliases", "")
            != resolved_row.get("matched_aliases", "")
            or tasks_before != tasks_after
        ):
            report_rows.append({
                "article_instance_key":
                    resolved_row.get("article_instance_key", ""),
                "title": resolved_row.get("title", ""),
                "full_name": resolved_row.get("full_name", ""),
                "author_index": resolved_row.get("author_index", ""),
                "aliases_before":
                    original_row.get("matched_aliases", ""),
                "aliases_after":
                    resolved_row.get("matched_aliases", ""),
                "tasks_before": tasks_before,
                "tasks_after": tasks_after,
                "tasks_removed": aligned["removed_tasks"],
                "uncertain_tasks": aligned["uncertain_tasks"],
                "task_alignment_status": aligned["status"],
                "postprocess_rules":
                    resolved_row.get("postprocess_rules", ""),
                "postprocess_confidence":
                    resolved_row.get("postprocess_confidence", ""),
            })

    with OUTPUT_CSV.open(
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=output_fields,
            extrasaction="ignore",
        )
        writer.writeheader()
        writer.writerows(output_rows)

    nonempty_rows = [
        row
        for row in output_rows
        if split_pipe(row.get("tasks", ""))
    ]



    return summary, len(output_rows), len(nonempty_rows), len(report_rows)


summary, output_count, nonempty_count, report_count = run_alignment()


print("TASK ALIGNMENT COMPLETE")
print("-" * 60)
print(f"Output rows: {output_count:,}")
print(f"Nonempty output rows: {nonempty_count:,}")
print(f"Rows in change report: {report_count:,}")
for key, value in summary.items():
    print(f"{key}: {value:,}")
print("-" * 60)



TASK ALIGNMENT COMPLETE
------------------------------------------------------------
Output rows: 211,085
Nonempty output rows: 210,646
Rows in change report: 3,050
rows: 211,085
rows_with_alias_changes: 3,020
rows_with_task_changes: 1,998
rows_with_tasks_removed: 1,995
tasks_removed: 3,273
------------------------------------------------------------
